# F3-modelo - Base comum da execução experimental

Este notebook estabelece a infraestrutura compartilhada por toda a Fase 3. Ele prepara, verifica e congela os elementos que deverão permanecer idênticos nas estratégias F3-A, F3-B, F3-C, F3-D e F3-E. Sua função não é produzir resultados experimentais do modelo aluno, mas garantir que todas as estratégias utilizem a mesma base de dados, os mesmos folds, os mesmos exemplos few-shot, os mesmos templates, o mesmo validador, as mesmas métricas e as mesmas configurações de inferência.

A F3-modelo recebe como entradas os pacotes operacionais concluídos nas fases anteriores:

- **F0**, com o CAMPI canônico, os cinco folds, a gramática do subconjunto Nile, o validador, as métricas e os templates;
- **F1**, com o esquema formal da IR, as 50 IRs de referência e as funções determinísticas de conversão;
- **F2**, com as 150 justificativas R1, R2 e R3 validadas e consolidadas.

O conjunto experimental é o **CAMPI**, formado por 50 pares entre uma intenção em linguagem natural, escrita em inglês, e sua expressão Nile de referência. O modelo aluno é o **Qwen2.5-1.5B-Instruct**, utilizado apenas em inferência, sem ajuste fino. Os cinco folds mantêm 40 exemplos de treino e 10 exemplos de teste por fold. Cada exemplo aparece no conjunto de teste de exatamente um fold.

As demonstrações few-shot são selecionadas exclusivamente entre os exemplos de treino do fold correspondente. São considerados `k=1`, `k=3` e `k=5`, com três métodos de seleção: aleatório determinístico, lexical e semântico. O regime `k=0` é utilizado apenas nas estratégias F3-A e F3-B. As listas são cumulativas: a seleção de `k=1` corresponde ao primeiro exemplo da ordenação; a de `k=3`, aos três primeiros; e a de `k=5`, aos cinco primeiros.

A grade principal contém 74 condições experimentais. Como cada condição é aplicada aos 50 exemplos do CAMPI ao longo dos cinco folds, são previstos 3.700 registros principais. A etapa complementar de autocorreção final poderá receber, no máximo, 1.850 registros elegíveis, dependendo das saídas inválidas produzidas nas estratégias que incluem autocorreção.

| Bloco | Etapa | Função metodológica | Artefato ou verificação principal |
|---:|---|---|---|
| 1 | Configuração geral | Fixar parâmetros, diretórios, sementes e dependências | Ambiente comum da F3 |
| 2 | Auditoria das entradas | Localizar e conferir F0, F1 e F2 | `f3_input_audit.csv` |
| 3 | Carga da base comum | Integrar CAMPI, folds, IRs, justificativas e templates | Estrutura unificada por exemplo |
| 4 | Núcleo formal | Verificar Nile, IR e métricas | Validação determinística das referências |
| 5 | Grade experimental | Materializar estratégias, variantes, métodos e valores de `k` | `f3_grid.csv` e esquema dos resultados |
| 6 | Seleção few-shot | Excluir equivalências formais e construir subconjuntos cumulativos | `f3_equivalencia_formal.csv`, `f3_selecao_fewshot.csv` e `f3_exemplos_fewshot.jsonl` |
| 7 | Montagem dos prompts | Padronizar demonstrações e auditar todos os testes | `f3_prompt_audit.csv` |
| 8 | Pós-processamento e avaliação | Fixar extração, validação e cálculo das métricas | Testes do esquema comum |
| 9 | Modelo aluno | Resolver a fonte do Qwen e fixar a inferência | Configuração de execução |
| 10 | Empacotamento | Congelar a base reutilizável pelas estratégias | `f3_modelo_operacional.zip` |

Ao final, o pacote operacional contém apenas a infraestrutura comum da Fase 3. Os pesos do modelo aluno e os resultados das estratégias não fazem parte deste notebook.

## Bloco 1 - Configuração geral

### Objetivo

Este bloco fixa os parâmetros que deverão ser mantidos em toda a Fase 3. A centralização dessas definições evita que cada estratégia adote valores diferentes para sementes, limites de geração, caminhos, nomes de arquivos ou métodos de seleção.

### Definições realizadas

O bloco:

- importa as bibliotecas utilizadas na preparação experimental;
- instala ou verifica dependências estritamente necessárias;
- identifica a fase, o conjunto CAMPI e o modelo aluno;
- define a semente global `42`;
- registra os cinco folds e a divisão de 40 exemplos de treino e 10 de teste;
- fixa os regimes `k=0`, `k=1`, `k=3` e `k=5`, respeitando as estratégias em que cada regime é permitido;
- define os métodos de seleção aleatório, lexical e semântico;
- fixa os limites máximos de novos tokens para saídas Nile e IR;
- define a inferência determinística, sem amostragem;
- organiza os diretórios temporários, operacionais e finais utilizados pelo notebook.

### Princípio de reprodutibilidade

A mesma configuração será exportada para os notebooks das estratégias. Alterações posteriores nesses parâmetros exigiriam a reconstrução da base comum, pois modificariam as condições experimentais.

### Resultado esperado

Ao final, todos os parâmetros e diretórios necessários aos blocos seguintes estarão disponíveis. Nenhuma tabela é exibida neste bloco, pois ele apenas inicializa o ambiente e não produz dados experimentais.

In [1]:
# ----------------------------------------------------------
# 1.1 Importação das bibliotecas principais
# ----------------------------------------------------------

import hashlib
import gc
import importlib
import importlib.metadata
import json
import os
import platform
import random
import re
import shutil
import subprocess
import sys
import time
import warnings
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import HTML, display

warnings.filterwarnings("ignore")


# ----------------------------------------------------------
# 1.2 Dependências operacionais
# ----------------------------------------------------------

INSTALAR_DEPENDENCIAS_AUSENTES = True

DEPENDENCIAS = {
    "lark": "lark>=1.1.9",
    "jsonschema": "jsonschema>=4.18",
    "sklearn": "scikit-learn>=1.3",
    "sentence_transformers": "sentence-transformers>=3.0",
    "transformers": "transformers>=4.37.0",
    "accelerate": "accelerate>=0.26.0",
}


def modulo_disponivel(nome):
    return importlib.util.find_spec(nome) is not None


faltantes = [
    pacote_pip
    for modulo, pacote_pip in DEPENDENCIAS.items()
    if not modulo_disponivel(modulo)
]

if faltantes and INSTALAR_DEPENDENCIAS_AUSENTES:
    print("Instalando dependências ausentes:", ", ".join(faltantes))
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        *faltantes,
    ])
    importlib.invalidate_caches()

faltantes_restantes = [
    modulo
    for modulo in DEPENDENCIAS
    if not modulo_disponivel(modulo)
]

if faltantes_restantes:
    raise ImportError(
        "Dependências ainda ausentes: "
        + ", ".join(faltantes_restantes)
    )


# ----------------------------------------------------------
# 1.3 Identificação da fase e do conjunto
# ----------------------------------------------------------

FASE = "F3-modelo"
DATASET_ID = "CAMPI"
TOTAL_EXEMPLOS = 50
TOTAL_FOLDS = 5
EXEMPLOS_TESTE_POR_FOLD = 10
EXEMPLOS_TREINO_POR_FOLD = 40
SEED = 42

MODELO_ALUNO = "Qwen2.5-1.5B-Instruct"
MODELO_ALUNO_HF = "Qwen/Qwen2.5-1.5B-Instruct"
MODELO_SEMANTICO_HF = "sentence-transformers/all-MiniLM-L6-v2"

K_FEWSHOT = [1, 3, 5]
METODOS_FEWSHOT = [
    "random_seed42",
    "lexical_tfidf",
    "semantic_embeddings",
]
METODO_ZERO_SHOT = "zero_shot"

ROTULOS_METODOS = {
    "zero_shot": "Nenhum (zero-shot)",
    "random_seed42": "Aleatória",
    "lexical_tfidf": "Lexical",
    "semantic_embeddings": "Semântica",
}

ORDEM_METODOS = {
    "zero_shot": 0,
    "random_seed42": 1,
    "lexical_tfidf": 2,
    "semantic_embeddings": 3,
}


# ----------------------------------------------------------
# 1.4 Configurações fixas de geração
# ----------------------------------------------------------

GERACAO = {
    "do_sample": False,
    "seed": SEED,
    "max_new_tokens_nile": 256,
    "max_new_tokens_ir": 512,
    "use_cache": True,
}

CARREGAR_MODELO_ALUNO_NA_F3_MODELO = False
EXECUTAR_TESTE_CURTO_MODELO_ALUNO = False
PERMITIR_DOWNLOAD_MODELOS = True


# ----------------------------------------------------------
# 1.5 Diretórios de trabalho
# ----------------------------------------------------------

KAGGLE_INPUT_DIR = Path("/kaggle/input")
KAGGLE_WORKING_DIR = Path("/kaggle/working")

if KAGGLE_WORKING_DIR.exists():
    BASE_DIR = KAGGLE_WORKING_DIR / "f3_modelo"
else:
    BASE_DIR = Path("/mnt/data/f3_modelo")

ENTRADAS_DIR = BASE_DIR / "entradas"
ARTEFATOS_DIR = BASE_DIR / "artefatos"
MODULOS_DIR = BASE_DIR / "modulos"
PACOTE_DIR = BASE_DIR / "pacote"

for diretorio in [
    BASE_DIR,
    ENTRADAS_DIR,
    ARTEFATOS_DIR,
    MODULOS_DIR,
    PACOTE_DIR,
]:
    diretorio.mkdir(parents=True, exist_ok=True)

F3_COMMON_PATH = MODULOS_DIR / "f3_common.py"
F3_CONFIG_PATH = ARTEFATOS_DIR / "f3_config.json"
F3_GRID_PATH = ARTEFATOS_DIR / "f3_grid.csv"
F3_INPUT_AUDIT_PATH = ARTEFATOS_DIR / "f3_input_audit.csv"
F3_SELECTION_DETAIL_PATH = ARTEFATOS_DIR / "f3_selecao_fewshot.csv"
F3_SELECTION_JSONL_PATH = ARTEFATOS_DIR / "f3_exemplos_fewshot.jsonl"
F3_FORMAL_EQUIVALENCE_PATH = ARTEFATOS_DIR / "f3_equivalencia_formal.csv"
F3_PROMPT_METADATA_PATH = ARTEFATOS_DIR / "f3_prompt_templates.json"
F3_PROMPT_AUDIT_PATH = ARTEFATOS_DIR / "f3_prompt_audit.csv"
F3_RESULT_SCHEMA_PATH = ARTEFATOS_DIR / "f3_result_schema.json"
F3_MANIFEST_PATH = PACOTE_DIR / "manifest.json"
F3_ZIP_PATH = BASE_DIR.parent / "f3_modelo_operacional.zip"

CONTADOR_TABELAS = 0

random.seed(SEED)
np.random.seed(SEED)

print(f"Fase: {FASE}")
print(f"Conjunto: {DATASET_ID}")
print(f"Modelo aluno: {MODELO_ALUNO}")
print(f"Diretório de trabalho: {BASE_DIR}")
print("Status: OK")

Fase: F3-modelo
Conjunto: CAMPI
Modelo aluno: Qwen2.5-1.5B-Instruct
Diretório de trabalho: /kaggle/working/f3_modelo
Status: OK


## Bloco 2 - Auditoria das entradas F0, F1 e F2

### Objetivo

Este bloco confirma que os três pacotes operacionais necessários à Fase 3 estão disponíveis e são compatíveis entre si. A identificação não depende somente do nome da fonte. O conteúdo interno e os manifestos são examinados para impedir o uso acidental de uma versão incompleta ou incompatível.

### Formatos aceitos no Kaggle

O Kaggle normalmente monta cada dataset como um diretório em `/kaggle/input`. Por isso, o bloco aceita duas formas equivalentes de entrada:

- **diretório operacional já extraído**, contendo `manifest.json` e os demais arquivos da fase;
- **arquivo ZIP operacional**, contendo os mesmos artefatos na raiz.

Exemplos válidos:

```text
/kaggle/input/f0-operacional/manifest.json
/kaggle/input/f1-operacional/manifest.json
/kaggle/input/f2-operacional/manifest.json
```

ou:

```text
/kaggle/input/.../f0_operacional.zip
/kaggle/input/.../f1_operacional.zip
/kaggle/input/.../f2_operacional.zip
```

Não é necessário compactar novamente os datasets que já aparecem expandidos no painel de entrada do Kaggle.

### Entradas esperadas

São procuradas fontes correspondentes a:

- F0, com CAMPI, folds, gramática, validador, métricas e templates;
- F1, com o esquema e as referências da IR;
- F2, com as justificativas R1, R2 e R3 consolidadas.

### Procedimentos

O bloco:

1. cria `f3_common.py`, módulo reutilizável com funções comuns à Fase 3;
2. procura diretórios e arquivos ZIP em `/kaggle/input`;
3. identifica a fase pelo campo `fase` do `manifest.json`;
4. verifica a presença dos arquivos obrigatórios;
5. seleciona uma cadeia compatível F0 → F1 → F2;
6. copia ou extrai cada fonte para uma área de trabalho controlada;
7. compara os hashes usados na rastreabilidade entre as fases;
8. registra a origem efetivamente utilizada.

### Barreiras de segurança

A execução é interrompida quando:

- alguma fase não é encontrada;
- um arquivo obrigatório está ausente;
- o manifesto não corresponde ao CAMPI;
- F1 não referencia a F0 carregada;
- F2 não referencia a F0 e a F1 carregadas;
- as 150 justificativas da F2 não estão aprovadas;
- a auditoria final automática da F2 não foi concluída.

### Resultado esperado

A tabela de auditoria informa, para cada fase, se a origem foi um diretório ou ZIP, o caminho localizado, a impressão digital da fonte, o hash do manifesto e o vínculo com a fase anterior. O arquivo `f3_input_audit.csv` preserva esse diagnóstico no pacote final.

In [2]:
# ----------------------------------------------------------
# 2.1 Criação do módulo comum da F3
# ----------------------------------------------------------

COMMON_MODULE_SOURCE = 'from __future__ import annotations\n\nimport csv\nimport hashlib\nimport importlib.util\nimport json\nimport os\nimport random\nimport re\nimport sys\nimport time\nimport zipfile\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple\n\n\ndef sha256_bytes(data: bytes) -> str:\n    return hashlib.sha256(data).hexdigest()\n\n\ndef sha256_text(text: str) -> str:\n    return sha256_bytes(str(text).encode("utf-8"))\n\n\ndef sha256_file(path: str | Path, chunk_size: int = 1024 * 1024) -> str:\n    path = Path(path)\n    digest = hashlib.sha256()\n    with path.open("rb") as handle:\n        for chunk in iter(lambda: handle.read(chunk_size), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef read_json(path: str | Path) -> Any:\n    with Path(path).open("r", encoding="utf-8") as handle:\n        return json.load(handle)\n\n\ndef write_json(data: Any, path: str | Path, *, indent: int = 2) -> None:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("w", encoding="utf-8") as handle:\n        json.dump(data, handle, ensure_ascii=False, indent=indent)\n        handle.write("\\n")\n\n\ndef read_jsonl(path: str | Path) -> List[Dict[str, Any]]:\n    records: List[Dict[str, Any]] = []\n    with Path(path).open("r", encoding="utf-8") as handle:\n        for line_number, line in enumerate(handle, start=1):\n            stripped = line.strip()\n            if not stripped:\n                continue\n            try:\n                record = json.loads(stripped)\n            except json.JSONDecodeError as exc:\n                raise ValueError(\n                    f"JSONL inválido em {path}, linha {line_number}: {exc}"\n                ) from exc\n            if not isinstance(record, dict):\n                raise TypeError(\n                    f"Registro JSONL em {path}, linha {line_number}, não é um objeto."\n                )\n            records.append(record)\n    return records\n\n\ndef write_jsonl(records: Iterable[Mapping[str, Any]], path: str | Path) -> None:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("w", encoding="utf-8") as handle:\n        for record in records:\n            handle.write(\n                json.dumps(dict(record), ensure_ascii=False, separators=(",", ":"))\n            )\n            handle.write("\\n")\n\n\ndef extract_zip(zip_path: str | Path, output_dir: str | Path, *, clean: bool = True) -> Path:\n    zip_path = Path(zip_path)\n    output_dir = Path(output_dir)\n    if clean and output_dir.exists():\n        import shutil\n        shutil.rmtree(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(zip_path, "r") as package:\n        package.extractall(output_dir)\n    return output_dir\n\n\ndef import_module_from_path(module_name: str, path: str | Path):\n    path = Path(path)\n    spec = importlib.util.spec_from_file_location(module_name, path)\n    if spec is None or spec.loader is None:\n        raise ImportError(f"Não foi possível criar o módulo {module_name} a partir de {path}.")\n    module = importlib.util.module_from_spec(spec)\n    sys.modules[module_name] = module\n    spec.loader.exec_module(module)\n    return module\n\n\ndef canonical_json(data: Any) -> str:\n    return json.dumps(\n        data,\n        ensure_ascii=False,\n        sort_keys=True,\n        separators=(",", ":"),\n    )\n\n\ndef stable_seed(base_seed: int, *parts: Any) -> int:\n    payload = "|".join([str(base_seed), *[str(part) for part in parts]])\n    return int(hashlib.sha256(payload.encode("utf-8")).hexdigest()[:16], 16) % (2**32)\n\n\ndef fill_template(template: str, values: Mapping[str, Any]) -> str:\n    result = str(template)\n    for key, value in values.items():\n        result = result.replace("{{" + str(key) + "}}", str(value))\n    unresolved = sorted(set(re.findall(r"\\{\\{([A-Z0-9_]+)\\}\\}", result)))\n    if unresolved:\n        raise ValueError(\n            "Placeholders não preenchidos: " + ", ".join(unresolved)\n        )\n    return result.strip() + "\\n"\n\n\ndef strip_code_fences(text: Any) -> str:\n    value = "" if text is None else str(text).strip()\n    value = re.sub(r"^\\s*```(?:json|text|nile)?\\s*", "", value, flags=re.I)\n    value = re.sub(r"\\s*```\\s*$", "", value)\n    return value.strip()\n\n\ndef extract_balanced_json(text: Any) -> Tuple[Optional[Dict[str, Any]], str, Optional[str]]:\n    cleaned = strip_code_fences(text)\n    starts = [idx for idx, char in enumerate(cleaned) if char == "{"]\n    for start in starts:\n        depth = 0\n        in_string = False\n        escape = False\n        for index in range(start, len(cleaned)):\n            char = cleaned[index]\n            if in_string:\n                if escape:\n                    escape = False\n                elif char == "\\\\":\n                    escape = True\n                elif char == \'"\':\n                    in_string = False\n                continue\n            if char == \'"\':\n                in_string = True\n            elif char == "{":\n                depth += 1\n            elif char == "}":\n                depth -= 1\n                if depth == 0:\n                    candidate = cleaned[start:index + 1]\n                    try:\n                        parsed = json.loads(candidate)\n                    except json.JSONDecodeError:\n                        break\n                    if isinstance(parsed, dict):\n                        return parsed, candidate, None\n                    return None, candidate, "O JSON extraído não é um objeto."\n    return None, "", "Nenhum objeto JSON balanceado e válido foi encontrado."\n\n\ndef extract_nile(text: Any, validator: Any = None) -> Dict[str, Any]:\n    raw = "" if text is None else str(text)\n    cleaned = strip_code_fences(raw)\n    cleaned = cleaned.replace("\\r\\n", "\\n").replace("\\r", "\\n").strip()\n\n    start_positions = [m.start() for m in re.finditer(r"(?i)\\bdefine\\s+intent\\b", cleaned)]\n    candidates: List[str] = []\n\n    for start in start_positions:\n        remainder = cleaned[start:].strip()\n        candidates.append(remainder)\n        lines = [line.strip() for line in remainder.splitlines() if line.strip()]\n        if lines:\n            candidates.append(lines[0])\n            for end in range(len(lines), 0, -1):\n                candidates.append(" ".join(lines[:end]).strip())\n\n    if not candidates and cleaned:\n        candidates.append(cleaned)\n\n    unique_candidates: List[str] = []\n    seen = set()\n    for candidate in candidates:\n        candidate = re.sub(r"\\s+", " ", candidate).strip()\n        candidate = re.sub(r"^(?:Nile|Saída Nile|Output)\\s*:\\s*", "", candidate, flags=re.I)\n        if candidate and candidate not in seen:\n            seen.add(candidate)\n            unique_candidates.append(candidate)\n\n    if validator is not None:\n        for candidate in unique_candidates:\n            result = validator.validate(candidate)\n            if result.get("syntax_valid", False):\n                return {\n                    "nile": candidate,\n                    "extraction_valid": True,\n                    "extraction_method": "first_syntax_valid_candidate",\n                    "validation": result,\n                }\n\n    fallback = unique_candidates[0] if unique_candidates else ""\n    validation = validator.validate(fallback) if validator is not None else None\n    return {\n        "nile": fallback,\n        "extraction_valid": bool(fallback),\n        "extraction_method": "first_candidate_fallback" if fallback else "empty",\n        "validation": validation,\n    }\n\n\ndef format_demo_a(record: Mapping[str, Any], index: int) -> str:\n    return (\n        f"Exemplo {index}:\\n"\n        f"Entrada em linguagem natural:\\n{record[\'nl\']}\\n"\n        f"Saída Nile:\\n{record[\'nile\']}"\n    )\n\n\ndef format_demo_c_ir(record: Mapping[str, Any], index: int) -> str:\n    return (\n        f"Exemplo {index}:\\n"\n        f"Entrada em linguagem natural:\\n{record[\'nl\']}\\n"\n        f"Representação Intermediária:\\n{canonical_json(record[\'ir\'])}"\n    )\n\n\ndef format_demo_c_ir_nile(record: Mapping[str, Any], index: int) -> str:\n    return (\n        f"Exemplo {index}:\\n"\n        f"Representação Intermediária:\\n{canonical_json(record[\'ir\'])}\\n"\n        f"Saída Nile:\\n{record[\'nile\']}"\n    )\n\n\ndef format_demo_d_r3(record: Mapping[str, Any], index: int) -> str:\n    return (\n        f"Exemplo {index}:\\n"\n        f"Entrada em linguagem natural:\\n{record[\'nl\']}\\n"\n        f"R3:\\n{record[\'r3\']}\\n"\n        f"Saída Nile:\\n{record[\'nile\']}"\n    )\n\n\ndef format_demo_e_r1(record: Mapping[str, Any], index: int) -> str:\n    return (\n        f"Exemplo {index}:\\n"\n        f"Entrada em linguagem natural:\\n{record[\'nl\']}\\n"\n        f"R1:\\n{record[\'r1\']}\\n"\n        f"Representação Intermediária:\\n{canonical_json(record[\'ir\'])}"\n    )\n\n\ndef format_demo_e_r2(record: Mapping[str, Any], index: int) -> str:\n    return (\n        f"Exemplo {index}:\\n"\n        f"Representação Intermediária:\\n{canonical_json(record[\'ir\'])}\\n"\n        f"R2:\\n{record[\'r2\']}\\n"\n        f"Saída Nile:\\n{record[\'nile\']}"\n    )\n\n\nDEMO_FORMATTERS = {\n    "A": format_demo_a,\n    "C_IR": format_demo_c_ir,\n    "C_IR_NILE": format_demo_c_ir_nile,\n    "D_R3": format_demo_d_r3,\n    "E_R1": format_demo_e_r1,\n    "E_R2": format_demo_e_r2,\n}\n\n\ndef build_demos(\n    selected_ids: Sequence[str],\n    records_by_id: Mapping[str, Mapping[str, Any]],\n    format_key: str,\n) -> str:\n    if not selected_ids:\n        return "Nenhum exemplo demonstrativo."\n    if format_key not in DEMO_FORMATTERS:\n        raise KeyError(f"Formato de demonstração desconhecido: {format_key}")\n    formatter = DEMO_FORMATTERS[format_key]\n    parts = []\n    for index, item_id in enumerate(selected_ids, start=1):\n        if item_id not in records_by_id:\n            raise KeyError(f"ID demonstrativo ausente: {item_id}")\n        parts.append(formatter(records_by_id[item_id], index))\n    return "\\n\\n".join(parts)\n\n\ndef build_prompt_a(template: str, nl_test: str, demos: str) -> str:\n    return fill_template(template, {\n        "EXEMPLOS_FEW_SHOT": demos,\n        "NL_TESTE": nl_test,\n    })\n\n\ndef build_prompt_c_ir(template: str, nl_test: str, demos: str, ir_schema: Mapping[str, Any]) -> str:\n    return fill_template(template, {\n        "IR_SCHEMA_JSON": json.dumps(ir_schema, ensure_ascii=False, indent=2),\n        "EXEMPLOS_FEW_SHOT_IR": demos,\n        "NL_TESTE": nl_test,\n    })\n\n\ndef build_prompt_c_nile(template: str, ir_test: Mapping[str, Any] | str, demos: str) -> str:\n    ir_text = ir_test if isinstance(ir_test, str) else canonical_json(ir_test)\n    return fill_template(template, {\n        "EXEMPLOS_FEW_SHOT_IR_NILE": demos,\n        "IR_TESTE": ir_text,\n    })\n\n\ndef build_prompt_d(template: str, nl_test: str, demos: str) -> str:\n    return fill_template(template, {\n        "EXEMPLOS_FEW_SHOT_R3": demos,\n        "NL_TESTE": nl_test,\n    })\n\n\ndef build_prompt_e_ir(template: str, nl_test: str, demos: str, ir_schema: Mapping[str, Any]) -> str:\n    return fill_template(template, {\n        "IR_SCHEMA_JSON": json.dumps(ir_schema, ensure_ascii=False, indent=2),\n        "EXEMPLOS_FEW_SHOT_R1": demos,\n        "NL_TESTE": nl_test,\n    })\n\n\ndef build_prompt_e_nile(template: str, ir_test: Mapping[str, Any] | str, demos: str) -> str:\n    ir_text = ir_test if isinstance(ir_test, str) else canonical_json(ir_test)\n    return fill_template(template, {\n        "EXEMPLOS_FEW_SHOT_R2": demos,\n        "IR_TESTE": ir_text,\n    })\n\n\ndef build_prompt_autocorrection(\n    template: str,\n    nl_test: str,\n    auxiliary_artifacts: str,\n    rejected_nile: str,\n    validator_feedback: str,\n    demos: str,\n) -> str:\n    return fill_template(template, {\n        "NL_TESTE": nl_test,\n        "ARTEFATOS_AUXILIARES": auxiliary_artifacts or "Nenhum.",\n        "NILE_REJEITADA": rejected_nile,\n        "FEEDBACK_VALIDACAO_NILE": validator_feedback,\n        "EXEMPLOS_FEW_SHOT": demos,\n    })\n\n\ndef load_selection_map(path: str | Path) -> Dict[Tuple[int, str, str, int], Dict[str, Any]]:\n    records = read_jsonl(path)\n    mapping: Dict[Tuple[int, str, str, int], Dict[str, Any]] = {}\n    for record in records:\n        key = (\n            int(record["fold"]),\n            str(record["test_id"]),\n            str(record["selection_method"]),\n            int(record["k"]),\n        )\n        if key in mapping:\n            raise ValueError(f"Seleção duplicada para {key}.")\n        mapping[key] = record\n    return mapping\n\n\ndef get_selected_ids(\n    selection_map: Mapping[Tuple[int, str, str, int], Mapping[str, Any]],\n    fold: int,\n    test_id: str,\n    selection_method: str,\n    k: int,\n) -> List[str]:\n    if int(k) == 0:\n        return []\n    key = (int(fold), str(test_id), str(selection_method), int(k))\n    if key not in selection_map:\n        raise KeyError(f"Seleção não encontrada: {key}")\n    return list(selection_map[key]["selected_ids"])\n\n\ndef resolve_local_model_dir(root: str | Path, name_hints: Sequence[str]) -> Optional[Path]:\n    root = Path(root)\n    if not root.exists():\n        return None\n    hints = [hint.lower().replace("_", "-") for hint in name_hints]\n    candidates = []\n    for config_path in root.rglob("config.json"):\n        parent = config_path.parent\n        normalized = str(parent).lower().replace("_", "-")\n        if all(hint in normalized for hint in hints):\n            candidates.append(parent)\n    if not candidates:\n        return None\n    candidates.sort(key=lambda path: (len(path.parts), str(path)))\n    return candidates[0]\n\n\ndef load_student_model(\n    model_source: str | Path,\n    *,\n    use_cuda: bool = True,\n    local_files_only: bool = False,\n):\n    try:\n        import torch\n        from transformers import AutoModelForCausalLM, AutoTokenizer\n    except ImportError as exc:\n        raise ImportError(\n            "Instale transformers e accelerate antes de carregar o modelo aluno."\n        ) from exc\n\n    source = str(model_source)\n    cuda_available = bool(use_cuda and torch.cuda.is_available())\n    dtype = torch.float16 if cuda_available else torch.float32\n\n    tokenizer = AutoTokenizer.from_pretrained(\n        source,\n        use_fast=True,\n        local_files_only=local_files_only,\n        trust_remote_code=False,\n    )\n    model = AutoModelForCausalLM.from_pretrained(\n        source,\n        torch_dtype=dtype,\n        device_map="auto" if cuda_available else None,\n        low_cpu_mem_usage=True,\n        local_files_only=local_files_only,\n        trust_remote_code=False,\n    )\n    if not cuda_available:\n        model.to("cpu")\n    model.eval()\n\n    if tokenizer.pad_token_id is None:\n        tokenizer.pad_token_id = tokenizer.eos_token_id\n\n    return tokenizer, model\n\n\ndef generate_student_output(\n    prompt: str,\n    tokenizer: Any,\n    model: Any,\n    *,\n    max_new_tokens: int,\n    seed: int = 42,\n) -> Dict[str, Any]:\n    import torch\n    try:\n        from transformers import set_seed\n        set_seed(seed)\n    except Exception:\n        random.seed(seed)\n        torch.manual_seed(seed)\n        if torch.cuda.is_available():\n            torch.cuda.manual_seed_all(seed)\n\n    messages = [{"role": "user", "content": prompt}]\n    encoded = tokenizer.apply_chat_template(\n        messages,\n        add_generation_prompt=True,\n        tokenize=True,\n        return_dict=True,\n        return_tensors="pt",\n    )\n    model_device = next(model.parameters()).device\n    encoded = {key: value.to(model_device) for key, value in encoded.items()}\n    prompt_tokens = int(encoded["input_ids"].shape[-1])\n\n    start = time.perf_counter()\n    with torch.inference_mode():\n        generated = model.generate(\n            **encoded,\n            do_sample=False,\n            max_new_tokens=int(max_new_tokens),\n            pad_token_id=tokenizer.pad_token_id,\n            eos_token_id=tokenizer.eos_token_id,\n            use_cache=True,\n        )\n    elapsed = time.perf_counter() - start\n\n    new_ids = generated[0, prompt_tokens:]\n    text = tokenizer.decode(new_ids, skip_special_tokens=True).strip()\n\n    return {\n        "response_raw": text,\n        "prompt_tokens": prompt_tokens,\n        "generated_tokens": int(new_ids.shape[-1]),\n        "elapsed_seconds": float(elapsed),\n        "finish_reason": "eos_or_limit",\n    }\n\n\ndef validate_generated_ir(text: Any, ir_schema: Mapping[str, Any], ir_core_module: Any) -> Dict[str, Any]:\n    parsed, extracted, extraction_error = extract_balanced_json(text)\n    if parsed is None:\n        return {\n            "ir": None,\n            "ir_text": extracted,\n            "ir_valid": False,\n            "ir_errors": [extraction_error or "Falha de extração da IR."],\n        }\n    result = ir_core_module.validate_ir(parsed, dict(ir_schema))\n    errors = [\n        f"{item.get(\'path\')}: {item.get(\'message\')}"\n        for item in result.get("schema_errors", [])\n    ]\n    return {\n        "ir": parsed,\n        "ir_text": extracted,\n        "ir_valid": bool(result.get("schema_valid", False)),\n        "ir_errors": errors,\n    }\n\n\ndef evaluate_nile_output(\n    reference_nile: str,\n    response_text: Any,\n    validator: Any,\n    metrics_module: Any,\n) -> Dict[str, Any]:\n    extraction = extract_nile(response_text, validator=validator)\n    predicted = extraction["nile"]\n    metrics = metrics_module.evaluate_pair(reference_nile, predicted, validator)\n    return {\n        **extraction,\n        **metrics,\n        "validator_feedback": metrics.get("prediction_feedback", ""),\n    }\n\n\ndef make_result_record(**kwargs: Any) -> Dict[str, Any]:\n    defaults = {\n        "phase": "F3",\n        "strategy": None,\n        "variant": None,\n        "fold": None,\n        "id": None,\n        "k": None,\n        "selection_method": None,\n        "selection_label": None,\n        "demonstration_ids": [],\n        "demonstration_scores": [],\n        "nl": None,\n        "reference_nile": None,\n        "prompt_primary": None,\n        "response_primary_raw": None,\n        "prompt_secondary": None,\n        "response_secondary_raw": None,\n        "prompt_autocorrection": None,\n        "response_autocorrection_raw": None,\n        "generated_ir": None,\n        "generated_ir_text": None,\n        "ir_valid": None,\n        "ir_errors": [],\n        "nile_initial": None,\n        "nile_corrected": None,\n        "nile_evaluated": None,\n        "syntax_valid": None,\n        "validator_feedback": None,\n        "autocorrection_applied": False,\n        "attempt": 1,\n        "psr": None,\n        "em": None,\n        "ed": None,\n        "ned": None,\n        "sla_s": None,\n        "sla_f": None,\n        "prompt_primary_sha256": None,\n        "prompt_secondary_sha256": None,\n        "prompt_autocorrection_sha256": None,\n        "prompt_primary_tokens": None,\n        "prompt_secondary_tokens": None,\n        "prompt_autocorrection_tokens": None,\n        "generated_primary_tokens": None,\n        "generated_secondary_tokens": None,\n        "generated_autocorrection_tokens": None,\n        "elapsed_primary_seconds": None,\n        "elapsed_secondary_seconds": None,\n        "elapsed_autocorrection_seconds": None,\n        "status": "pending",\n        "error": None,\n    }\n    unexpected = sorted(set(kwargs) - set(defaults))\n    if unexpected:\n        raise KeyError("Campos de resultado desconhecidos: " + ", ".join(unexpected))\n    defaults.update(kwargs)\n    hash_pairs = [\n        ("prompt_primary", "prompt_primary_sha256"),\n        ("prompt_secondary", "prompt_secondary_sha256"),\n        ("prompt_autocorrection", "prompt_autocorrection_sha256"),\n    ]\n    for prompt_field, hash_field in hash_pairs:\n        if defaults[prompt_field] is not None and defaults[hash_field] is None:\n            defaults[hash_field] = sha256_text(defaults[prompt_field])\n    return defaults\n'

F3_COMMON_PATH.write_text(
    COMMON_MODULE_SOURCE,
    encoding="utf-8",
)

if str(MODULOS_DIR) not in sys.path:
    sys.path.insert(0, str(MODULOS_DIR))

import f3_common


# ----------------------------------------------------------
# 2.2 Funções de exibição e localização
# ----------------------------------------------------------


ROTULOS_COLUNAS_F3 = {
    "id": "ID",
    "test_id": "ID de teste",
    "dataset": "conjunto de dados",
    "tipo_fonte": "tipo de fonte",
    "caminho": "caminho",
    "caminho localizado": "caminho localizado",
    "manifesto_sha256": "SHA-256 do manifesto",
    "fingerprint": "SHA-256 da origem",
    "sha256": "SHA-256",
    "status": "status",
    "fase": "fase",
    "arquivo": "arquivo",
    "origem": "origem",
    "registros": "registros",
    "estratégia": "estratégia",
    "fluxo": "fluxo",
    "configurações": "configurações",
    "registros esperados": "registros esperados",
    "item": "item",
    "total": "total",
    "método": "método",
    "k": "k",
    "fold": "fold",
    "posição": "posição",
    "id selecionado": "ID selecionado",
    "id de teste": "ID de teste",
    "vazamento do teste": "vazamento do teste",
    "pertence ao treino": "pertence ao treino",
    "métrica": "métrica",
    "campo interno": "campo interno",
    "melhor valor": "melhor valor",
    "observação": "observação",
}


def traduzir_valores_para_exibicao_f3(df):
    tabela = df.copy()

    def converter_valor(valor):
        if isinstance(valor, (bool, np.bool_)):
            return "sim" if valor else "não"
        if valor is None:
            return "-"
        try:
            if not isinstance(valor, str) and pd.isna(valor):
                return "-"
        except Exception:
            pass
        if isinstance(valor, str):
            valor_limpo = valor.strip()
            valor_lower = valor_limpo.lower()
            if valor_lower == "true":
                return "sim"
            if valor_lower == "false":
                return "não"
            if valor_lower in {
                "nan", "none", "null", "nat", "n/a",
                "não se aplica", "nao se aplica",
            }:
                return "-"
        return valor

    for coluna in tabela.columns:
        tabela[coluna] = tabela[coluna].apply(converter_valor)

    return tabela.rename(
        columns={
            coluna: ROTULOS_COLUNAS_F3.get(
                coluna,
                str(coluna).replace("_", " "),
            )
            for coluna in tabela.columns
        }
    )


def exibir_tabela(
    df,
    titulo=None,
    altura_px=None,
    largura_px=None,
    mostrar_indice=False,
):
    global CONTADOR_TABELAS

    if df is None:
        print("Tabela não disponível.")
        return

    if not isinstance(df, pd.DataFrame):
        df = pd.DataFrame(df)

    if df.empty:
        print("Tabela vazia.")
        return

    CONTADOR_TABELAS += 1
    tabela = traduzir_valores_para_exibicao_f3(df)

    titulo_final = f"Tabela {CONTADOR_TABELAS}"
    if titulo:
        titulo_final += f". {titulo}"

    html_tabela = tabela.to_html(
        escape=True,
        index=mostrar_indice,
        border=0,
        justify="left",
        classes="tabela_saida",
    )

    largura_css = (
        f"{largura_px}px"
        if largura_px is not None
        else "100%"
    )

    altura_css = (
        "overflow-y: visible;"
        if altura_px is None
        else f"max-height: {altura_px}px; overflow-y: auto;"
    )

    display(HTML(f"""
    <div style="
        font-weight: 600;
        font-size: 15px;
        margin-top: 8px;
        margin-bottom: 6px;
        color: #f1f1f1;
    ">
        {titulo_final}
    </div>

    <div class="container_tabela_saida" style="
        display: inline-block;
        width: {largura_css};
        max-width: 100%;
        overflow-x: auto;
        {altura_css}
        box-sizing: border-box;
        padding: 0;
        margin-top: 8px;
        margin-bottom: 12px;
        border: none;
        border-radius: 0;
    ">
        <style>
            .container_tabela_saida {{
                box-sizing: border-box !important;
            }}

            .container_tabela_saida table.tabela_saida {{
                border-collapse: collapse !important;
                table-layout: auto !important;
                width: auto !important;
                min-width: unset !important;
                max-width: none !important;
                font-family: Arial, sans-serif !important;
                font-size: 13px !important;
                background-color: #111 !important;
                color: #f1f1f1 !important;
                border: 1px solid #555 !important;
            }}

            .container_tabela_saida table.tabela_saida thead th {{
                position: sticky !important;
                top: 0 !important;
                z-index: 2 !important;
                background-color: #2b2b2b !important;
                color: #ffffff !important;
                font-weight: bold !important;
                padding: 7px !important;
                text-align: left !important;
                white-space: nowrap !important;
                border: 1px solid #777 !important;
                border-bottom: 2px solid #888 !important;
            }}

            .container_tabela_saida table.tabela_saida tbody td {{
                padding: 7px !important;
                vertical-align: top !important;
                text-align: left !important;
                white-space: nowrap !important;
                border: 1px solid #555 !important;
            }}

            .container_tabela_saida table.tabela_saida
            tbody tr:nth-child(even) td {{
                background-color: #1b1b1b !important;
            }}

            .container_tabela_saida table.tabela_saida
            tbody tr:nth-child(odd) td {{
                background-color: #111 !important;
            }}
        </style>

        {html_tabela}
    </div>
    """))


RAIZES_BUSCA = [
    KAGGLE_INPUT_DIR,
    Path("/mnt/data"),
    Path.cwd(),
]


def listar_arquivos_relativos(diretorio):
    diretorio = Path(diretorio)
    return sorted(
        caminho.relative_to(diretorio).as_posix()
        for caminho in diretorio.rglob("*")
        if caminho.is_file()
    )


def sha256_diretorio(diretorio):
    """
    Calcula uma impressão digital determinística para um diretório.

    O hash incorpora o caminho relativo e o conteúdo de cada arquivo.
    Ele é usado apenas para auditoria da origem montada pelo Kaggle.
    Os vínculos metodológicos entre F0, F1 e F2 continuam sendo
    verificados pelos hashes dos manifestos e dos artefatos internos.
    """
    diretorio = Path(diretorio)
    digest = hashlib.sha256()

    for caminho_relativo in listar_arquivos_relativos(diretorio):
        caminho = diretorio / caminho_relativo
        digest.update(caminho_relativo.encode("utf-8"))
        digest.update(b"\0")

        with caminho.open("rb") as arquivo:
            for bloco in iter(lambda: arquivo.read(1024 * 1024), b""):
                digest.update(bloco)

        digest.update(b"\0")

    return digest.hexdigest()


def ler_fonte_operacional(caminho, tipo_fonte):
    caminho = Path(caminho)

    if tipo_fonte == "zip":
        with zipfile.ZipFile(caminho, "r") as pacote:
            nomes = sorted(pacote.namelist())

            if "manifest.json" not in nomes:
                raise FileNotFoundError(
                    "manifest.json ausente na raiz do ZIP."
                )

            manifesto_bytes = pacote.read("manifest.json")

        fingerprint = f3_common.sha256_file(caminho)

    elif tipo_fonte == "diretorio":
        manifesto_path = caminho / "manifest.json"

        if not manifesto_path.is_file():
            raise FileNotFoundError(
                "manifest.json ausente no diretório operacional."
            )

        nomes = listar_arquivos_relativos(caminho)
        manifesto_bytes = manifesto_path.read_bytes()
        fingerprint = sha256_diretorio(caminho)

    else:
        raise ValueError(
            f"Tipo de fonte operacional desconhecido: {tipo_fonte}"
        )

    manifesto = json.loads(
        manifesto_bytes.decode("utf-8")
    )

    return {
        "manifesto": manifesto,
        "manifesto_bytes": manifesto_bytes,
        "manifesto_sha256": f3_common.sha256_bytes(
            manifesto_bytes
        ),
        "nomes": nomes,
        "fingerprint": fingerprint,
    }


def validar_manifesto_fase(fase, manifesto, nomes_fonte):
    erros = []

    if manifesto.get("fase") != fase:
        erros.append(f"fase declarada diferente de {fase}")

    if fase == "F0":
        dataset = manifesto.get("dataset", {})

        if dataset.get("id") != DATASET_ID:
            erros.append("dataset da F0 diferente de CAMPI")

        if dataset.get("total_examples") != TOTAL_EXEMPLOS:
            erros.append(
                "quantidade de exemplos da F0 diferente de 50"
            )

        metricas = manifesto.get(
            "metrics",
            {},
        ).get("names", [])

        if metricas != [
            "PSR",
            "EM",
            "ED",
            "NED",
            "SLA-S",
            "SLA-F",
        ]:
            erros.append(
                "métricas da F0 diferentes do padrão vigente"
            )

        obrigatorios = {
            "campi_canonical.csv",
            "folds.csv",
            "nile_subset.lark",
            "nile_core.py",
            "nile_metrics.py",
            "biblioteca_prompts.json",
        }

    elif fase == "F1":
        if manifesto.get(
            "input",
            {},
        ).get("dataset") != DATASET_ID:
            erros.append(
                "dataset de entrada da F1 diferente de CAMPI"
            )

        ir = manifesto.get("ir", {})

        if ir.get("total_examples") != TOTAL_EXEMPLOS:
            erros.append(
                "quantidade de IRs diferente de 50"
            )

        if ir.get(
            "schema_valid_examples"
        ) != TOTAL_EXEMPLOS:
            erros.append(
                "nem todas as IRs foram aprovadas pelo esquema"
            )

        obrigatorios = {
            "ir_core.py",
            "ir_references.jsonl",
            "ir_schema.json",
            "ir_validation.csv",
        }

    elif fase == "F2":
        if manifesto.get("dataset") != DATASET_ID:
            erros.append(
                "dataset da F2 diferente de CAMPI"
            )

        rationales = manifesto.get("rationales", {})

        if rationales.get("total") != 150:
            erros.append(
                "quantidade de justificativas diferente de 150"
            )

        if not rationales.get("all_approved", False):
            erros.append(
                "as justificativas da F2 não estão todas aprovadas"
            )

        validation = manifesto.get("validation", {})

        if validation.get(
            "manual_qualitative_review_performed"
        ) is not False:
            erros.append(
                "manifesto da F2 declara revisão manual incompatível"
            )

        if validation.get(
            "automatic_final_audit_completed"
        ) is not True:
            erros.append(
                "auditoria final automática da F2 não concluída"
            )

        obrigatorios = {
            "rationale_core.py",
            "rationales_references.jsonl",
            "rationale_validation.csv",
        }

        arquivos_geracoes_professor = {
            "professor_generations.jsonl.gz",
            "professor_generations.jsonl",
        }

        if not (
            arquivos_geracoes_professor
            & set(nomes_fonte)
        ):
            erros.append(
                "arquivo ausente: "
                "professor_generations.jsonl.gz "
                "ou professor_generations.jsonl"
            )

    else:
        raise ValueError(
            f"Fase desconhecida: {fase}"
        )

    ausentes = sorted(
        obrigatorios - set(nomes_fonte)
    )

    if ausentes:
        erros.append(
            "arquivos ausentes: "
            + ", ".join(ausentes)
        )

    return erros


def descobrir_fontes_operacionais():
    """
    Descobre duas formas aceitas de entrada:

    1. diretório já extraído pelo Kaggle, como
       /kaggle/input/f0-operacional/manifest.json;
    2. arquivo ZIP operacional, como f0_operacional.zip.

    O Kaggle normalmente monta datasets como diretórios. Por isso,
    a busca por diretórios é necessária e tem preferência sobre ZIPs.
    """
    fontes = []
    chaves_vistas = set()

    for raiz in RAIZES_BUSCA:
        raiz = Path(raiz)

        if not raiz.exists():
            continue

        # Diretórios operacionais já extraídos.
        for manifesto_path in raiz.rglob("manifest.json"):
            diretorio = manifesto_path.parent
            chave = ("diretorio", str(diretorio.resolve()))

            if chave in chaves_vistas:
                continue

            chaves_vistas.add(chave)
            fontes.append({
                "caminho": diretorio,
                "tipo_fonte": "diretorio",
            })

        # Pacotes operacionais ainda compactados.
        for caminho_zip in raiz.rglob("*.zip"):
            chave = ("zip", str(caminho_zip.resolve()))

            if chave in chaves_vistas:
                continue

            chaves_vistas.add(chave)
            fontes.append({
                "caminho": caminho_zip,
                "tipo_fonte": "zip",
            })

    return fontes


def listar_fontes_operacionais_validas(fase):
    padrao = fase.lower() + "_operacional"
    candidatos = []

    for fonte in descobrir_fontes_operacionais():
        caminho = fonte["caminho"]
        tipo_fonte = fonte["tipo_fonte"]

        try:
            dados = ler_fonte_operacional(
                caminho,
                tipo_fonte,
            )
            manifesto = dados["manifesto"]

            # Manifestos de outra fase são ignorados sem poluir
            # o relatório de erro da fase procurada.
            if manifesto.get("fase") != fase:
                continue

            erros = validar_manifesto_fase(
                fase,
                manifesto,
                dados["nomes"],
            )

            manifesto_sha256 = dados[
                "manifesto_sha256"
            ]
            fingerprint = dados["fingerprint"]

        except Exception as exc:
            # Só registra falhas de fontes cujo nome sugere a fase.
            nome_normalizado = (
                caminho.name.lower().replace("-", "_")
            )

            if padrao not in nome_normalizado:
                continue

            manifesto = None
            manifesto_sha256 = None
            fingerprint = None
            erros = [f"falha de leitura: {exc}"]

        nome_normalizado = (
            caminho.name.lower().replace("-", "_")
        )
        nome_exato = nome_normalizado in {
            padrao,
            f"{padrao}.zip",
        }

        candidatos.append({
            "caminho": caminho,
            "tipo_fonte": tipo_fonte,
            "manifesto": manifesto,
            "manifesto_sha256": manifesto_sha256,
            "fingerprint": fingerprint,
            "erros": erros,
            "valido": len(erros) == 0,
            "mtime": caminho.stat().st_mtime,
            "nome_exato": nome_exato,
        })

    validos = [
        item
        for item in candidatos
        if item["valido"]
    ]

    validos.sort(
        key=lambda item: (
            item["tipo_fonte"] == "diretorio",
            item["nome_exato"],
            item["mtime"],
            str(item["caminho"]),
        ),
        reverse=True,
    )

    if not validos:
        detalhes = "\n".join(
            (
                f"- {item['caminho']} "
                f"({item['tipo_fonte']}): "
                f"{'; '.join(item['erros'])}"
            )
            for item in candidatos
        ) or (
            "Nenhum diretório ou ZIP operacional "
            "foi encontrado."
        )

        raise FileNotFoundError(
            f"Nenhuma fonte operacional válida para {fase}.\n"
            f"{detalhes}"
        )

    return validos


def selecionar_cadeia_compativel():
    candidatos_f0 = listar_fontes_operacionais_validas(
        "F0"
    )
    candidatos_f1 = listar_fontes_operacionais_validas(
        "F1"
    )
    candidatos_f2 = listar_fontes_operacionais_validas(
        "F2"
    )

    cadeias = []

    for f2 in candidatos_f2:
        f2_inputs = f2["manifesto"].get(
            "inputs",
            {},
        )

        for f1 in candidatos_f1:
            f1_input = f1["manifesto"].get(
                "input",
                {},
            )

            f1_files = {
                item.get("arquivo"): item.get("sha256")
                for item in f1["manifesto"].get(
                    "files",
                    [],
                )
            }

            f1_manifest_exact = (
                f2_inputs.get("f1_manifest_sha256")
                == f1["manifesto_sha256"]
            )

            f1_content_exact = (
                f2_inputs.get("ir_references_sha256")
                == f1_files.get("ir_references.jsonl")
                and f2_inputs.get("ir_schema_sha256")
                == f1_files.get("ir_schema.json")
            )

            if not (
                f1_manifest_exact
                or f1_content_exact
            ):
                continue

            for f0 in candidatos_f0:
                if (
                    f1_input.get("manifest_sha256")
                    != f0["manifesto_sha256"]
                ):
                    continue

                if (
                    f2_inputs.get("f0_manifest_sha256")
                    != f0["manifesto_sha256"]
                ):
                    continue

                cadeias.append({
                    "F0": f0,
                    "F1": f1,
                    "F2": f2,
                    "f1_link_mode": (
                        "manifest_sha256"
                        if f1_manifest_exact
                        else (
                            "ir_schema_and_"
                            "references_sha256"
                        )
                    ),
                })

    if not cadeias:
        resumo = {
            "F0": [
                item["manifesto_sha256"]
                for item in candidatos_f0
            ],
            "F1": [
                {
                    "manifesto": item[
                        "manifesto_sha256"
                    ],
                    "f0_referenciado": item[
                        "manifesto"
                    ].get(
                        "input",
                        {},
                    ).get("manifest_sha256"),
                }
                for item in candidatos_f1
            ],
            "F2": [
                {
                    "manifesto": item[
                        "manifesto_sha256"
                    ],
                    "entradas": item[
                        "manifesto"
                    ].get("inputs", {}),
                }
                for item in candidatos_f2
            ],
        }

        raise ValueError(
            "Nenhuma cadeia compatível F0 -> F1 -> F2 "
            "foi encontrada. "
            + json.dumps(
                resumo,
                ensure_ascii=False,
            )
        )

    cadeias.sort(
        key=lambda chain: (
            chain[
                "f1_link_mode"
            ] == "manifest_sha256",
            sum(
                chain[fase][
                    "tipo_fonte"
                ] == "diretorio"
                for fase in ["F0", "F1", "F2"]
            ),
            chain["F2"]["mtime"],
            chain["F1"]["mtime"],
            chain["F0"]["mtime"],
        ),
        reverse=True,
    )

    return cadeias[0]


def materializar_fonte_operacional(item, destino):
    caminho = Path(item["caminho"])
    destino = Path(destino)

    if destino.exists():
        shutil.rmtree(destino)

    if item["tipo_fonte"] == "zip":
        f3_common.extract_zip(
            caminho,
            destino,
            clean=False,
        )

    elif item["tipo_fonte"] == "diretorio":
        shutil.copytree(
            caminho,
            destino,
        )

    else:
        raise ValueError(
            "Tipo de fonte operacional não suportado: "
            f"{item['tipo_fonte']}"
        )

    if not (
        destino / "manifest.json"
    ).is_file():
        raise FileNotFoundError(
            f"manifest.json não foi materializado para "
            f"{destino.name}."
        )


# ----------------------------------------------------------
# 2.3 Localização e materialização das fontes
# ----------------------------------------------------------

pacotes = selecionar_cadeia_compativel()

DIRETORIOS_FASES = {}

for fase in ["F0", "F1", "F2"]:
    item = pacotes[fase]
    destino_fase = ENTRADAS_DIR / fase.lower()

    materializar_fonte_operacional(
        item,
        destino_fase,
    )

    DIRETORIOS_FASES[fase] = destino_fase

F0_DIR = DIRETORIOS_FASES["F0"]
F1_DIR = DIRETORIOS_FASES["F1"]
F2_DIR = DIRETORIOS_FASES["F2"]

MANIFESTO_F0 = pacotes["F0"]["manifesto"]
MANIFESTO_F1 = pacotes["F1"]["manifesto"]
MANIFESTO_F2 = pacotes["F2"]["manifesto"]

HASH_MANIFESTO_F0 = f3_common.sha256_file(
    F0_DIR / "manifest.json"
)
HASH_MANIFESTO_F1 = f3_common.sha256_file(
    F1_DIR / "manifest.json"
)
HASH_MANIFESTO_F2 = f3_common.sha256_file(
    F2_DIR / "manifest.json"
)

if (
    MANIFESTO_F1["input"]["manifest_sha256"]
    != HASH_MANIFESTO_F0
):
    raise ValueError(
        "A F1 não referencia o manifesto carregado da F0."
    )

if (
    MANIFESTO_F2["inputs"]["f0_manifest_sha256"]
    != HASH_MANIFESTO_F0
):
    raise ValueError(
        "A F2 não referencia o manifesto carregado da F0."
    )

F1_MANIFEST_LINK_EXACT = (
    MANIFESTO_F2["inputs"]["f1_manifest_sha256"]
    == HASH_MANIFESTO_F1
)

F1_CORE_LINK_EXACT = (
    MANIFESTO_F2["inputs"]["ir_references_sha256"]
    == f3_common.sha256_file(
        F1_DIR / "ir_references.jsonl"
    )
    and MANIFESTO_F2["inputs"]["ir_schema_sha256"]
    == f3_common.sha256_file(
        F1_DIR / "ir_schema.json"
    )
)

if not (
    F1_MANIFEST_LINK_EXACT
    or F1_CORE_LINK_EXACT
):
    raise ValueError(
        "A F2 não referencia os artefatos carregados da F1."
    )


# ----------------------------------------------------------
# 2.4 Relatório de auditoria das entradas
# ----------------------------------------------------------

input_audit_rows = []

for fase in ["F0", "F1", "F2"]:
    item = pacotes[fase]
    diretorio = DIRETORIOS_FASES[fase]

    input_audit_rows.append({
        "fase": fase,
        "tipo de fonte": item["tipo_fonte"],
        "origem": item["caminho"].name,
        "caminho localizado": str(item["caminho"]),
        "sha256 da origem": item["fingerprint"],
        "sha256 do manifesto": f3_common.sha256_file(
            diretorio / "manifest.json"
        ),
        "vínculo com a fase anterior": (
            pacotes.get(
                "f1_link_mode",
                "manifest_sha256",
            )
            if fase == "F1"
            else "manifest_sha256"
        ),
        "status": "compatível",
    })

input_audit_df = pd.DataFrame(
    input_audit_rows
)

input_audit_df.to_csv(
    F3_INPUT_AUDIT_PATH,
    index=False,
    encoding="utf-8",
)

exibir_tabela(
    input_audit_df,
    "Fontes operacionais localizadas e auditadas",
    altura_px=300,
)

print(
    "Compatibilidade entre F0, F1 e F2: confirmada"
)
print(
    "Formatos aceitos: diretório montado pelo Kaggle ou ZIP."
)
print("Status: OK")

fase,tipo de fonte,origem,caminho localizado,sha256 da origem,sha256 do manifesto,vínculo com a fase anterior,status
F0,diretorio,f0-operacional,/kaggle/input/datasets/thiagoarajoguedes/f0-operacional,2a12b9f6f7a5d11add79d3f4035a6d8b60a65dcdcd0d3369a9ccf2cc1f93e368,53f33d7efd81d2f63243c467e6f67c151f52f4c330935cb13488ba05f8ba4f3d,manifest_sha256,compatível
F1,diretorio,f1-operacional,/kaggle/input/datasets/thiagoarajoguedes/f1-operacional,422ca2b5d05cedd95b076554e86cd0bbe0b824b0185c374a2cc57c328b2d220a,e15ad59c7161dce8bb93ebe7cd3cae38199b869ba724cb87aef2eddc4f043df9,manifest_sha256,compatível
F2,diretorio,f2-operacional,/kaggle/input/datasets/thiagoarajoguedes/f2-operacional,21bf0e44f14d5bb99b2fb617c35103fbc6c5cc7bfbf293542ed6e845106ba2bc,095d65515ed044e252f42d1254219316bde605be560120052bc6a7962e7444c6,manifest_sha256,compatível


Compatibilidade entre F0, F1 e F2: confirmada
Formatos aceitos: diretório montado pelo Kaggle ou ZIP.
Status: OK


## Bloco 3 - Carga da base comum

### Objetivo

Este bloco transforma os artefatos separados de F0, F1 e F2 em uma base unificada por exemplo. Essa estrutura permitirá que os notebooks das estratégias recuperem, por um único identificador, todos os elementos necessários à montagem das demonstrações e à avaliação das saídas.

### Artefatos carregados

Para cada um dos 50 IDs do CAMPI, são carregados:

- a intenção em linguagem natural, escrita em inglês;
- a expressão Nile de referência;
- o fold e o split associados;
- a IR de referência produzida na F1;
- as justificativas R1, R2 e R3 consolidadas na F2.

Também são carregados:

- os cinco folds completos;
- o esquema JSON da IR;
- os templates utilizados pelas estratégias e pela autocorreção;
- os metadados dos prompts.

### Verificações de integridade

O bloco confirma que:

- existem exatamente 50 registros canônicos;
- existem exatamente 50 IRs;
- existem exatamente 50 conjuntos completos de R1, R2 e R3;
- os IDs são únicos e coincidem entre F0, F1 e F2;
- cada fold contém 40 exemplos de treino e 10 de teste;
- cada ID aparece no teste de apenas um fold;
- não há justificativas, IRs ou referências sem correspondência no CAMPI;
- todos os templates necessários à Fase 3 estão disponíveis.

### Estrutura unificada

A estrutura `records_by_id` reúne os artefatos de cada exemplo. Ela é utilizada apenas como fonte controlada de dados. Durante a execução experimental, os artefatos de referência do exemplo de teste permanecem indisponíveis ao modelo aluno e são acessados somente na etapa de avaliação.

### Saída

Ao final, o notebook exibe o resumo da carga e a cobertura dos artefatos. Qualquer inconsistência impede o avanço para a montagem da grade e dos prompts.

In [3]:
# ----------------------------------------------------------
# 3.1 Caminhos dos artefatos carregados
# ----------------------------------------------------------

CAMPI_PATH = F0_DIR / "campi_canonical.csv"
FOLDS_PATH = F0_DIR / "folds.csv"
GRAMMAR_PATH = F0_DIR / "nile_subset.lark"
NILE_CORE_PATH = F0_DIR / "nile_core.py"
NILE_METRICS_PATH = F0_DIR / "nile_metrics.py"
PROMPT_LIBRARY_PATH = F0_DIR / "biblioteca_prompts.json"
PROMPTS_DIR = F0_DIR / "prompts"

IR_CORE_PATH = F1_DIR / "ir_core.py"
IR_REFERENCES_PATH = F1_DIR / "ir_references.jsonl"
IR_SCHEMA_PATH = F1_DIR / "ir_schema.json"

RATIONALES_PATH = F2_DIR / "rationales_references.jsonl"


# ----------------------------------------------------------
# 3.2 Leitura dos artefatos
# ----------------------------------------------------------

campi_df = pd.read_csv(CAMPI_PATH)
folds_df = pd.read_csv(FOLDS_PATH)
ir_records = f3_common.read_jsonl(IR_REFERENCES_PATH)
rationale_records = f3_common.read_jsonl(RATIONALES_PATH)
ir_schema = f3_common.read_json(IR_SCHEMA_PATH)
prompt_library = f3_common.read_json(PROMPT_LIBRARY_PATH)

ir_by_id = {item["id"]: item["ir"] for item in ir_records}
rationales_by_id = {
    item["id"]: {
        "r1": item["r1"],
        "r2": item["r2"],
        "r3": item["r3"],
    }
    for item in rationale_records
}

ids_campi = set(campi_df["id"].astype(str))
ids_ir = set(ir_by_id)
ids_rationales = set(rationales_by_id)
ids_folds = set(folds_df["id"].astype(str))

if len(campi_df) != TOTAL_EXEMPLOS:
    raise ValueError("O CAMPI canônico não contém 50 exemplos.")

if not (ids_campi == ids_ir == ids_rationales == ids_folds):
    raise ValueError("Os identificadores de CAMPI, folds, IR e justificativas divergem.")

if campi_df["id"].duplicated().any():
    raise ValueError("O CAMPI contém IDs duplicados.")

if len(folds_df) != TOTAL_EXEMPLOS * TOTAL_FOLDS:
    raise ValueError("O arquivo de folds não contém 250 registros.")

fold_summary = (
    folds_df.groupby(["fold", "split"], as_index=False)
    .size()
    .pivot(index="fold", columns="split", values="size")
    .reset_index()
)

if set(fold_summary["fold"]) != set(range(1, TOTAL_FOLDS + 1)):
    raise ValueError("Os folds esperados de 1 a 5 não foram encontrados.")

if not (
    (fold_summary["test"] == EXEMPLOS_TESTE_POR_FOLD).all()
    and (fold_summary["train"] == EXEMPLOS_TREINO_POR_FOLD).all()
):
    raise ValueError("Algum fold não possui 40 exemplos de treino e 10 de teste.")

contagem_teste = (
    folds_df.loc[folds_df["split"] == "test"]
    .groupby("id")
    .size()
)

if not (contagem_teste == 1).all():
    raise ValueError("Cada exemplo deve aparecer exatamente uma vez como teste.")


# ----------------------------------------------------------
# 3.3 Carga dos templates da Fase 3
# ----------------------------------------------------------

PROMPT_IDS_F3 = [
    "f3a_traducao_direta",
    "f3c_gerar_ir_teste",
    "f3c_gerar_nile_a_partir_ir",
    "f3d_traducao_com_r3",
    "f3e_gerar_ir_com_r1",
    "f3e_gerar_nile_com_r2",
    "f3_autocorrecao_nile",
]

prompt_templates = {}
prompt_metadata = {}

for prompt_id in PROMPT_IDS_F3:
    metadata = prompt_library["prompts"].get(prompt_id)
    if metadata is None:
        raise KeyError(f"Template ausente na biblioteca: {prompt_id}")

    caminho = F0_DIR / metadata["arquivo_relativo"]
    if not caminho.is_file():
        raise FileNotFoundError(caminho)

    texto = caminho.read_text(encoding="utf-8")
    placeholders_reais = sorted(
        set(re.findall(r"\{\{([A-Z0-9_]+)\}\}", texto))
    )
    placeholders_declarados = sorted(metadata["placeholders"])

    if placeholders_reais != placeholders_declarados:
        raise ValueError(
            f"Placeholders divergentes em {prompt_id}: "
            f"reais={placeholders_reais}; "
            f"declarados={placeholders_declarados}"
        )

    prompt_templates[prompt_id] = texto
    prompt_metadata[prompt_id] = {
        **metadata,
        "sha256": f3_common.sha256_file(caminho),
        "caracteres": len(texto),
    }


# ----------------------------------------------------------
# 3.4 Estrutura unificada por exemplo
# ----------------------------------------------------------

records_by_id = {}
for row in campi_df.itertuples(index=False):
    item_id = str(row.id)
    records_by_id[item_id] = {
        "id": item_id,
        "nl": str(row.nl),
        "nile": str(row.nile_canonical),
        "ir": ir_by_id[item_id],
        "r1": rationales_by_id[item_id]["r1"],
        "r2": rationales_by_id[item_id]["r2"],
        "r3": rationales_by_id[item_id]["r3"],
        "primary_family": str(row.primary_family),
        "has_temporal": bool(row.has_temporal),
        "has_route": bool(row.has_route),
        "complexity_score": int(row.complexity_score),
    }

summary_rows = [
    {"grupo": "CAMPI", "item": "Exemplos canônicos", "total": len(campi_df), "status": "OK"},
    {"grupo": "Folds", "item": "Partições", "total": folds_df["fold"].nunique(), "status": "OK"},
    {"grupo": "Folds", "item": "Registros treino/teste", "total": len(folds_df), "status": "OK"},
    {"grupo": "F1", "item": "IRs", "total": len(ir_records), "status": "OK"},
    {"grupo": "F2", "item": "Justificativas R1/R2/R3", "total": len(rationale_records) * 3, "status": "OK"},
    {"grupo": "F0", "item": "Templates da F3", "total": len(prompt_templates), "status": "OK"},
]

exibir_tabela(
    pd.DataFrame(summary_rows),
    "Resumo da carga da base comum",
    altura_px=360,
)

prompt_table = pd.DataFrame([
    {
        "prompt": prompt_id,
        "estratégia": metadata["estrategia"],
        "placeholders": ", ".join(metadata["placeholders"]),
        "caracteres": metadata["caracteres"],
        "status": "OK",
    }
    for prompt_id, metadata in prompt_metadata.items()
])

exibir_tabela(
    prompt_table,
    "Templates destinados ao modelo aluno",
    altura_px=420,
)

print("IDs e cobertura dos folds: consistentes")
print("Status: OK")

grupo,item,total,status
CAMPI,Exemplos canônicos,50,OK
Folds,Partições,5,OK
Folds,Registros treino/teste,250,OK
F1,IRs,50,OK
F2,Justificativas R1/R2/R3,150,OK
F0,Templates da F3,7,OK


prompt,estratégia,placeholders,caracteres,status
f3a_traducao_direta,A,"EXEMPLOS_FEW_SHOT, NL_TESTE",1714,OK
f3c_gerar_ir_teste,C,"IR_SCHEMA_JSON, EXEMPLOS_FEW_SHOT_IR, NL_TESTE",1969,OK
f3c_gerar_nile_a_partir_ir,C,"EXEMPLOS_FEW_SHOT_IR_NILE, IR_TESTE",1718,OK
f3d_traducao_com_r3,D,"EXEMPLOS_FEW_SHOT_R3, NL_TESTE",1856,OK
f3e_gerar_ir_com_r1,E,"IR_SCHEMA_JSON, EXEMPLOS_FEW_SHOT_R1, NL_TESTE",2089,OK
f3e_gerar_nile_com_r2,E,"EXEMPLOS_FEW_SHOT_R2, IR_TESTE",1847,OK
f3_autocorrecao_nile,B_C_D_E,"NL_TESTE, ARTEFATOS_AUXILIARES, NILE_REJEITADA, FEEDBACK_VALIDACAO_NILE, EXEMPLOS_FEW_SHOT",2057,OK


IDs e cobertura dos folds: consistentes
Status: OK


## Bloco 4 - Núcleo formal e verificações determinísticas

### Objetivo

Este bloco confirma que os componentes formais preparados nas fases anteriores continuam operacionais no ambiente da Fase 3. A validação é realizada antes de qualquer geração do modelo aluno para garantir que eventuais falhas posteriores não sejam confundidas com problemas da gramática, da IR ou das métricas.

### Componentes importados

São carregados diretamente dos pacotes congelados:

- a gramática Lark do subconjunto Nile;
- o módulo `nile_core.py`;
- o módulo `nile_metrics.py`;
- o módulo `ir_core.py`;
- o esquema `ir_schema.json`.

### Verificação das referências Nile

Cada uma das 50 expressões de referência é submetida à cadeia formal:

```text
Nile canônica → parser Lark → AST
```

A taxa de sucesso do parser deve ser 50/50. A validade sintática é representada por PSR. A aderência à referência é avaliada posteriormente por SLA-S e SLA-F.

### Verificação das IRs

Cada IR é:

1. validada contra o esquema JSON;
2. convertida determinística e internamente para a representação esperada;
3. comparada com a AST derivada da Nile correspondente.

Essa verificação assegura que as referências da F1 continuam alinhadas às expressões Nile do CAMPI.

### Teste das métricas

Cada referência é comparada consigo mesma. O resultado esperado é:

- `PSR = 1`;
- `EM = 1`;
- `ED = 0`;
- `NED = 0`;
- `SLA-S = 1`;
- `SLA-F = 1`.

Esse procedimento não mede desempenho experimental. Ele apenas testa se a implementação das métricas produz o comportamento determinístico esperado em um caso de identidade.

### Resultado esperado

O bloco deve confirmar 50 referências Nile válidas, 50 IRs válidas e ausência de divergências entre Nile, AST e IR. Qualquer falha interrompe a preparação da F3.

In [4]:
# ----------------------------------------------------------
# 4.1 Importação controlada dos módulos formais
# ----------------------------------------------------------

nile_core = f3_common.import_module_from_path(
    "f3_nile_core",
    NILE_CORE_PATH,
)
nile_metrics = f3_common.import_module_from_path(
    "f3_nile_metrics",
    NILE_METRICS_PATH,
)
ir_core = f3_common.import_module_from_path(
    "f3_ir_core",
    IR_CORE_PATH,
)

grammar_text = GRAMMAR_PATH.read_text(encoding="utf-8")
validator = nile_core.NileValidator(grammar_text)


# ----------------------------------------------------------
# 4.2 Validação das referências e das IRs
# ----------------------------------------------------------

formal_rows = []

for item_id in sorted(records_by_id):
    record = records_by_id[item_id]
    nile_validation = validator.validate(record["nile"])
    ir_validation = ir_core.validate_ir(record["ir"], ir_schema)
    metrics = nile_metrics.evaluate_pair(
        record["nile"],
        record["nile"],
        validator,
    )

    ast_from_nile = nile_validation.get("ast")
    ast_from_ir = ir_core.ir_to_ast(record["ir"])
    ast_equivalent = (
        nile_metrics.normalized_ast(ast_from_nile)
        == nile_metrics.normalized_ast(ast_from_ir)
    )
    rendered = nile_core.render_ast(ast_from_ir)

    formal_rows.append({
        "id": item_id,
        "sintaxe Nile": bool(nile_validation["syntax_valid"]),
        "IR válida": bool(ir_validation["schema_valid"]),
        "AST equivalente": bool(ast_equivalent),
        "roundtrip Nile": rendered == record["nile"],
        "PSR": metrics["psr"],
        "EM": metrics["em"],
        "ED": metrics["ed"],
        "NED": metrics["ned"],
        "SLA-S": metrics["sla_s"],
        "SLA-F": metrics["sla_f"],
    })

formal_df = pd.DataFrame(formal_rows)

boolean_columns = [
    "sintaxe Nile",
    "IR válida",
    "AST equivalente",
    "roundtrip Nile",
]

if not formal_df[boolean_columns].all().all():
    falhas = formal_df.loc[
        ~formal_df[boolean_columns].all(axis=1)
    ]
    raise ValueError(
        "Falha na verificação formal para: "
        + ", ".join(falhas["id"].tolist())
    )

if not (
    (formal_df["PSR"] == 1).all()
    and (formal_df["EM"] == 1).all()
    and (formal_df["ED"] == 0).all()
    and (formal_df["NED"] == 0).all()
    and (formal_df["SLA-S"] == 1).all()
    and (formal_df["SLA-F"] == 1).all()
):
    raise ValueError("O teste determinístico das métricas falhou.")

formal_summary = pd.DataFrame([
    {"verificação": "Referências com sintaxe válida", "aprovadas": int(formal_df["sintaxe Nile"].sum()), "total": TOTAL_EXEMPLOS, "status": "OK"},
    {"verificação": "IRs válidas", "aprovadas": int(formal_df["IR válida"].sum()), "total": TOTAL_EXEMPLOS, "status": "OK"},
    {"verificação": "ASTs equivalentes", "aprovadas": int(formal_df["AST equivalente"].sum()), "total": TOTAL_EXEMPLOS, "status": "OK"},
    {"verificação": "Roundtrip preservado", "aprovadas": int(formal_df["roundtrip Nile"].sum()), "total": TOTAL_EXEMPLOS, "status": "OK"},
    {"verificação": "Autocomparações métricas", "aprovadas": len(formal_df), "total": TOTAL_EXEMPLOS, "status": "OK"},
])

exibir_tabela(
    formal_summary,
    "Validação formal e teste determinístico das métricas",
    altura_px=360,
)

print("Núcleo formal carregado a partir dos artefatos congelados")
print("Status: OK")

verificação,aprovadas,total,status
Referências com sintaxe válida,50,50,OK
IRs válidas,50,50,OK
ASTs equivalentes,50,50,OK
Roundtrip preservado,50,50,OK
Autocomparações métricas,50,50,OK


Núcleo formal carregado a partir dos artefatos congelados
Status: OK


## Bloco 5 - Grade experimental e padrão dos resultados

### Objetivo

Este bloco materializa todas as condições que deverão ser executadas nas estratégias. A grade funciona como contrato experimental: ela define quais combinações de estratégia, variante, valor de `k` e método de seleção precisam produzir resultados.

### Estratégias representadas

- **F3-A**: tradução direta `NL → Nile`, sem autocorreção;
- **F3-B**: primeira autocorreção das saídas produzidas pela F3-A;
- **F3-C**: tradução em duas etapas `NL → IR → Nile`, antes e depois da primeira autocorreção da saída Nile;
- **F3-D**: tradução direta com demonstrações acompanhadas de R3, antes e depois da primeira autocorreção;
- **F3-E**: geração de IR com R1 e geração de Nile com R2, antes e depois da primeira autocorreção.

A autocorreção final não é incluída como nova estratégia. Ela será uma etapa complementar aplicada somente às saídas ainda inválidas de F3-B, F3-C, F3-D e F3-E.

### Formação das 74 condições

Para F3-A e F3-B:

- `k=0` utiliza somente o regime zero-shot;
- `k=1`, `k=3` e `k=5` utilizam os três métodos few-shot;
- cada uma possui 10 condições.

Para F3-C, F3-D e F3-E:

- não existe `k=0`;
- cada estratégia possui duas variantes, sem autocorreção e após a primeira autocorreção;
- cada variante combina três valores de `k` com três métodos de seleção;
- cada estratégia possui 18 condições.

O total é:

```text
10 + 10 + 18 + 18 + 18 = 74 condições
```

Cada condição cobre os 50 exemplos do CAMPI, totalizando 3.700 registros principais.

### Esquema comum dos resultados

O bloco também define o esquema JSON que deverá ser respeitado pelos notebooks das estratégias. Entre os campos registrados estão:

- estratégia, variante, fold, ID, `k` e método de seleção;
- IDs e escores das demonstrações;
- prompt e resposta bruta de cada etapa;
- IR gerada, quando aplicável;
- Nile inicial e Nile avaliada;
- validade sintática e feedback do parser;
- PSR, EM, ED, NED, SLA-S e SLA-F;
- dados de tempo, tokens e estado da execução.

### Artefatos produzidos

São produzidos `f3_grid.csv` e `f3_result_schema.json`. Os notebooks das estratégias deverão seguir esses dois artefatos sem alterar a definição das condições ou dos campos.

### Resultado esperado

A grade deve conter 74 condições, 3.700 registros planejados e um schema comum compatível com as seis métricas oficiais e os campos operacionais da Fase 3.

In [5]:
# ----------------------------------------------------------
# 5.1 Definição das estratégias e variantes
# ----------------------------------------------------------

STRATEGY_PLAN = {
    "F3-A": {
        "fluxo": "NL -> Nile",
        "variantes": ["sem_ac"],
        "valores_k": [0, 1, 3, 5],
    },
    "F3-B": {
        "fluxo": "1ª AC das saídas da F3-A",
        "variantes": ["primeira_ac"],
        "valores_k": [0, 1, 3, 5],
    },
    "F3-C": {
        "fluxo": "NL -> IR -> Nile",
        "variantes": ["sem_ac", "primeira_ac"],
        "valores_k": [1, 3, 5],
    },
    "F3-D": {
        "fluxo": "NL + R3 -> Nile",
        "variantes": ["sem_ac", "primeira_ac"],
        "valores_k": [1, 3, 5],
    },
    "F3-E": {
        "fluxo": "NL + R1 -> IR; IR + R2 -> Nile",
        "variantes": ["sem_ac", "primeira_ac"],
        "valores_k": [1, 3, 5],
    },
}


grid_rows = []
for strategy, plan in STRATEGY_PLAN.items():
    for variant in plan["variantes"]:
        for k in plan["valores_k"]:
            methods = [METODO_ZERO_SHOT] if k == 0 else METODOS_FEWSHOT
            for method in methods:
                grid_rows.append({
                    "estratégia": strategy,
                    "fluxo": plan["fluxo"],
                    "variante": variant,
                    "regime": "zero-shot" if k == 0 else "few-shot",
                    "método de seleção": method,
                    "rótulo do método": ROTULOS_METODOS[method],
                    "k": k,
                    "total de exemplos": TOTAL_EXEMPLOS,
                    "registros esperados": TOTAL_EXEMPLOS,
                })

grid_df = pd.DataFrame(grid_rows)
grid_df["ordem do método"] = grid_df["método de seleção"].map(ORDEM_METODOS)
grid_df = grid_df.sort_values(
    ["estratégia", "variante", "k", "ordem do método"]
).drop(columns="ordem do método").reset_index(drop=True)

grid_df.to_csv(F3_GRID_PATH, index=False, encoding="utf-8")

TOTAL_CONFIGURACOES_PRINCIPAIS = len(grid_df)
TOTAL_REGISTROS_PRINCIPAIS = int(grid_df["registros esperados"].sum())
TOTAL_REGISTROS_AC_FINAL_MAXIMO = 1850

if TOTAL_CONFIGURACOES_PRINCIPAIS != 74:
    raise ValueError("A grade principal deve conter 74 configurações.")

if TOTAL_REGISTROS_PRINCIPAIS != 3700:
    raise ValueError("A execução principal deve produzir 3.700 registros.")


# ----------------------------------------------------------
# 5.2 Esquema comum dos resultados detalhados
# ----------------------------------------------------------

nullable_string = {"type": ["string", "null"]}
nullable_integer = {"type": ["integer", "null"]}
nullable_number = {"type": ["number", "null"]}
nullable_boolean = {"type": ["boolean", "null"]}

result_fields = {
    "phase": {"type": "string", "const": "F3"},
    "strategy": nullable_string,
    "variant": nullable_string,
    "fold": nullable_integer,
    "id": nullable_string,
    "k": nullable_integer,
    "selection_method": nullable_string,
    "selection_label": nullable_string,
    "demonstration_ids": {"type": "array", "items": {"type": "string"}},
    "demonstration_scores": {"type": "array", "items": {"type": ["number", "null"]}},
    "nl": nullable_string,
    "reference_nile": nullable_string,
    "prompt_primary": nullable_string,
    "response_primary_raw": nullable_string,
    "prompt_secondary": nullable_string,
    "response_secondary_raw": nullable_string,
    "prompt_autocorrection": nullable_string,
    "response_autocorrection_raw": nullable_string,
    "generated_ir": {"type": ["object", "null"]},
    "generated_ir_text": nullable_string,
    "ir_valid": nullable_boolean,
    "ir_errors": {"type": "array", "items": {"type": "string"}},
    "nile_initial": nullable_string,
    "nile_corrected": nullable_string,
    "nile_evaluated": nullable_string,
    "syntax_valid": nullable_boolean,
    "validator_feedback": nullable_string,
    "autocorrection_applied": {"type": "boolean"},
    "attempt": {"type": "integer", "minimum": 1},
    "psr": nullable_integer,
    "em": nullable_integer,
    "ed": nullable_integer,
    "ned": nullable_number,
    "sla_s": nullable_integer,
    "sla_f": nullable_number,
    "prompt_primary_sha256": nullable_string,
    "prompt_secondary_sha256": nullable_string,
    "prompt_autocorrection_sha256": nullable_string,
    "prompt_primary_tokens": nullable_integer,
    "prompt_secondary_tokens": nullable_integer,
    "prompt_autocorrection_tokens": nullable_integer,
    "generated_primary_tokens": nullable_integer,
    "generated_secondary_tokens": nullable_integer,
    "generated_autocorrection_tokens": nullable_integer,
    "elapsed_primary_seconds": nullable_number,
    "elapsed_secondary_seconds": nullable_number,
    "elapsed_autocorrection_seconds": nullable_number,
    "status": {"type": "string"},
    "error": nullable_string,
}

RESULT_SCHEMA = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "title": "Registro detalhado da Fase 3",
    "type": "object",
    "properties": result_fields,
    "required": list(result_fields),
    "additionalProperties": False,
}

f3_common.write_json(RESULT_SCHEMA, F3_RESULT_SCHEMA_PATH)

strategy_summary = (
    grid_df.groupby(["estratégia", "fluxo"], as_index=False)
    .agg(
        configurações=("registros esperados", "size"),
        registros=("registros esperados", "sum"),
    )
)

exibir_tabela(
    strategy_summary,
    "Grade principal por estratégia",
    altura_px=360,
)

counts_df = pd.DataFrame([
    {"item": "Configurações principais", "total": TOTAL_CONFIGURACOES_PRINCIPAIS, "status": "OK"},
    {"item": "Registros principais", "total": TOTAL_REGISTROS_PRINCIPAIS, "status": "OK"},
    {"item": "Elegíveis máximos à AC final", "total": TOTAL_REGISTROS_AC_FINAL_MAXIMO, "status": "OK"},
    {"item": "Métricas por registro", "total": 6, "status": "PSR, EM, ED, NED, SLA-S e SLA-F"},
])

exibir_tabela(
    counts_df,
    "Totais esperados da Fase 3",
    altura_px=300,
)

print("Grade experimental e esquema de resultados congelados")
print("Status: OK")

estratégia,fluxo,configurações,registros
F3-A,NL -> Nile,10,500
F3-B,1ª AC das saídas da F3-A,10,500
F3-C,NL -> IR -> Nile,18,900
F3-D,NL + R3 -> Nile,18,900
F3-E,NL + R1 -> IR; IR + R2 -> Nile,18,900


item,total,status
Configurações principais,74,OK
Registros principais,3700,OK
Elegíveis máximos à AC final,1850,OK
Métricas por registro,6,"PSR, EM, ED, NED, SLA-S e SLA-F"


Grade experimental e esquema de resultados congelados
Status: OK


## Bloco 6 - Seleção cumulativa dos exemplos few-shot

### Objetivo

Este bloco determina, de maneira reproduzível, quais exemplos de treino serão usados como demonstrações para cada exemplo de teste. A seleção é realizada separadamente em cada fold e utiliza somente registros pertencentes ao split de treino.

### Controle de equivalência formal

A separação por ID não é suficiente quando dois exemplos diferentes possuem a mesma saída formal. Para impedir que a resposta de um exemplo de teste seja apresentada ao modelo aluno por meio de outro ID, o bloco calcula uma assinatura formal para cada referência.

A assinatura é derivada da AST normalizada e desconsidera apenas o identificador da intenção. As regras de normalização preservam a ordem dos middleboxes e ordenam somente os componentes cuja ordem não altera a intenção. Exemplos com a mesma assinatura são reunidos em um grupo formal.

Para cada exemplo de teste, são inelegíveis:

- o próprio ID do teste;
- qualquer exemplo com Nile de referência idêntica;
- qualquer exemplo com IR de referência idêntica;
- qualquer exemplo com a mesma assinatura formal normalizada.

A referência é usada somente nessa etapa de deduplicação e nunca é inserida no prompt do modelo aluno.

### Métodos de seleção

#### Aleatório determinístico

A lista de candidatos elegíveis é embaralhada por uma semente derivada da semente global, do fold e do ID de teste.

#### Lexical

Os textos do conjunto de treino são representados com TF-IDF, utilizando unigramas e bigramas. A ordenação lexical é calculada pela similaridade de cosseno e, em seguida, os exemplos formalmente inelegíveis são removidos.

#### Semântico

Os textos são codificados pelo modelo `sentence-transformers/all-MiniLM-L6-v2`. A ordenação utiliza o produto entre embeddings normalizados e também é filtrada pelo conjunto formalmente elegível.

### Regra cumulativa

Cada método produz uma única ordenação dos cinco melhores candidatos elegíveis:

```text
k=1 → posição 1
k=3 → posições 1, 2 e 3
k=5 → posições 1, 2, 3, 4 e 5
```

Os subconjuntos permanecem aninhados, permitindo comparar o aumento de `k` sem trocar os exemplos já utilizados nas condições menores.

### Rastreabilidade

O bloco registra:

- grupo formal do teste e da demonstração;
- IDs excluídos por equivalência formal;
- tamanho do universo original e do universo elegível;
- presença de Nile ou IR idêntica;
- método, posição, escore e semente;
- confirmação de pertencimento ao treino;
- confirmação de ausência de vazamento.

### Saídas

São produzidos:

- `f3_equivalencia_formal.csv`, com os grupos formais;
- `f3_selecao_fewshot.csv`, com as cinco posições de cada ordenação;
- `f3_exemplos_fewshot.jsonl`, com as listas cumulativas para `k=1`, `k=3` e `k=5`.

A execução falha se qualquer demonstração for formalmente equivalente ao teste, estiver fora do treino, repetir um ID ou violar a cumulatividade.

In [6]:
# ----------------------------------------------------------
# 6.1 Identificação das equivalências formais
# ----------------------------------------------------------

all_ids = sorted(records_by_id)
all_texts = [records_by_id[item_id]["nl"] for item_id in all_ids]
index_by_id = {
    item_id: index
    for index, item_id in enumerate(all_ids)
}


def normalizar_texto_nile_referencia(texto):
    return " ".join(str(texto).split())


nile_reference_by_id = {
    item_id: normalizar_texto_nile_referencia(
        records_by_id[item_id]["nile"]
    )
    for item_id in all_ids
}

ir_reference_by_id = {
    item_id: f3_common.canonical_json({
        chave: valor
        for chave, valor in records_by_id[item_id]["ir"].items()
        if chave != "intent_id"
    })
    for item_id in all_ids
}

formal_signature_by_id = {}

for item_id in all_ids:
    ast_referencia = ir_core.ir_to_ast(
        records_by_id[item_id]["ir"]
    )
    ast_normalizada = nile_metrics.normalized_ast(
        ast_referencia
    )
    formal_signature_by_id[item_id] = (
        f3_common.canonical_json(ast_normalizada)
    )

formal_signature_groups = {}

for item_id in all_ids:
    formal_signature_groups.setdefault(
        formal_signature_by_id[item_id],
        [],
    ).append(item_id)

formal_group_by_id = {}
formal_group_members = {}

group_items = sorted(
    formal_signature_groups.items(),
    key=lambda item: tuple(sorted(item[1])),
)

for group_index, (_, member_ids) in enumerate(
    group_items,
    start=1,
):
    group_id = f"formal_{group_index:03d}"
    members = sorted(member_ids)
    formal_group_members[group_id] = members

    for item_id in members:
        formal_group_by_id[item_id] = group_id

formal_equivalence_rows = []

for item_id in all_ids:
    group_id = formal_group_by_id[item_id]
    members = formal_group_members[group_id]
    equivalents = [
        candidate_id
        for candidate_id in members
        if candidate_id != item_id
    ]

    formal_equivalence_rows.append({
        "id": item_id,
        "grupo formal": group_id,
        "tamanho do grupo": len(members),
        "possui equivalente formal": bool(equivalents),
        "ids formalmente equivalentes": (
            ", ".join(equivalents)
            if equivalents
            else "-"
        ),
        "SHA-256 da Nile de referência": (
            f3_common.sha256_text(
                nile_reference_by_id[item_id]
            )
        ),
        "SHA-256 da IR de referência": (
            f3_common.sha256_text(
                ir_reference_by_id[item_id]
            )
        ),
        "SHA-256 da assinatura formal": (
            f3_common.sha256_text(
                formal_signature_by_id[item_id]
            )
        ),
    })

formal_equivalence_df = pd.DataFrame(
    formal_equivalence_rows
)

formal_equivalence_df.to_csv(
    F3_FORMAL_EQUIVALENCE_PATH,
    index=False,
    encoding="utf-8",
)

duplicate_formal_groups = {
    group_id: member_ids
    for group_id, member_ids in formal_group_members.items()
    if len(member_ids) > 1
}

FORMAL_EQUIVALENCE_INFO = {
    "method": (
        "normalized_ast_with_order_rules_"
        "and_without_intent_id"
    ),
    "groups_total": len(formal_group_members),
    "duplicate_groups": len(duplicate_formal_groups),
    "examples_in_duplicate_groups": int(
        sum(
            len(member_ids)
            for member_ids in duplicate_formal_groups.values()
        )
    ),
    "duplicate_group_members": duplicate_formal_groups,
    "exclusion_rule": (
        "A training example is ineligible when its "
        "normalized formal signature is identical to "
        "the test reference signature."
    ),
}

formal_duplicate_preview = formal_equivalence_df.loc[
    formal_equivalence_df[
        "possui equivalente formal"
    ]
].copy()

if not formal_duplicate_preview.empty:
    exibir_tabela(
        formal_duplicate_preview,
        "Exemplos com equivalência formal",
        altura_px=360,
    )
else:
    print(
        "Nenhum grupo com mais de um exemplo "
        "formalmente equivalente foi identificado."
    )


# ----------------------------------------------------------
# 6.2 Preparação do modelo semântico
# ----------------------------------------------------------

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

local_semantic_dir = f3_common.resolve_local_model_dir(
    KAGGLE_INPUT_DIR,
    ["all-minilm-l6-v2"],
)

if local_semantic_dir is not None:
    semantic_model_source = str(local_semantic_dir)
    semantic_local = True
else:
    semantic_model_source = MODELO_SEMANTICO_HF
    semantic_local = False

    if not PERMITIR_DOWNLOAD_MODELOS:
        raise FileNotFoundError(
            "O modelo semântico não foi encontrado "
            "em /kaggle/input. Adicione "
            "all-MiniLM-L6-v2 como dataset ou "
            "permita download."
        )

semantic_model = SentenceTransformer(
    semantic_model_source
)

semantic_embeddings = semantic_model.encode(
    all_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

if semantic_embeddings.shape[0] != TOTAL_EXEMPLOS:
    raise ValueError(
        "Quantidade inesperada de embeddings semânticos."
    )


# ----------------------------------------------------------
# 6.3 Construção das ordenações elegíveis
# ----------------------------------------------------------

selection_detail_rows = []
selection_jsonl_records = []

for fold in range(1, TOTAL_FOLDS + 1):
    train_ids = sorted(
        folds_df.loc[
            (folds_df["fold"] == fold)
            & (folds_df["split"] == "train"),
            "id",
        ].astype(str)
    )
    test_ids = sorted(
        folds_df.loc[
            (folds_df["fold"] == fold)
            & (folds_df["split"] == "test"),
            "id",
        ].astype(str)
    )

    if len(train_ids) != EXEMPLOS_TREINO_POR_FOLD:
        raise ValueError(
            f"Fold {fold} sem 40 exemplos de treino."
        )

    if len(test_ids) != EXEMPLOS_TESTE_POR_FOLD:
        raise ValueError(
            f"Fold {fold} sem 10 exemplos de teste."
        )

    train_texts = [
        records_by_id[item_id]["nl"]
        for item_id in train_ids
    ]

    tfidf = TfidfVectorizer(
        ngram_range=(1, 2),
        lowercase=True,
        sublinear_tf=True,
        norm="l2",
        token_pattern=r"(?u)\b\w+\b",
    )
    train_matrix = tfidf.fit_transform(train_texts)

    for test_id in test_ids:
        test_text = records_by_id[test_id]["nl"]
        test_group = formal_group_by_id[test_id]

        excluded_formal_ids = sorted([
            candidate_id
            for candidate_id in train_ids
            if formal_group_by_id[candidate_id] == test_group
        ])

        eligible_ids = [
            candidate_id
            for candidate_id in train_ids
            if candidate_id not in excluded_formal_ids
        ]

        if len(eligible_ids) < max(K_FEWSHOT):
            raise ValueError(
                "Universo elegível insuficiente para "
                f"{test_id} no fold {fold}: "
                f"{len(eligible_ids)} candidatos."
            )

        random_ids = eligible_ids.copy()
        random_seed = f3_common.stable_seed(
            SEED,
            "random_seed42",
            fold,
            test_id,
        )
        random.Random(random_seed).shuffle(random_ids)

        rankings = {
            "random_seed42": [
                (item_id, None)
                for item_id in random_ids
            ],
        }

        test_vector = tfidf.transform([test_text])
        lexical_scores = cosine_similarity(
            test_vector,
            train_matrix,
        )[0]

        lexical_ranking_complete = sorted(
            zip(
                train_ids,
                lexical_scores.tolist(),
            ),
            key=lambda pair: (-pair[1], pair[0]),
        )

        rankings["lexical_tfidf"] = [
            pair
            for pair in lexical_ranking_complete
            if pair[0] in eligible_ids
        ]

        test_embedding = semantic_embeddings[
            index_by_id[test_id]
        ]
        train_indices = [
            index_by_id[item_id]
            for item_id in train_ids
        ]

        scores_semantic = (
            semantic_embeddings[train_indices]
            @ test_embedding
        )

        semantic_ranking_complete = sorted(
            zip(
                train_ids,
                scores_semantic.tolist(),
            ),
            key=lambda pair: (-pair[1], pair[0]),
        )

        rankings["semantic_embeddings"] = [
            pair
            for pair in semantic_ranking_complete
            if pair[0] in eligible_ids
        ]

        for method in METODOS_FEWSHOT:
            ranking = rankings[method]

            if len(ranking) < max(K_FEWSHOT):
                raise ValueError(
                    "Ordenação elegível insuficiente para "
                    f"{fold}, {test_id}, {method}."
                )

            top_five = ranking[:max(K_FEWSHOT)]

            for rank, (selected_id, score) in enumerate(
                top_five,
                start=1,
            ):
                same_nile = (
                    nile_reference_by_id[selected_id]
                    == nile_reference_by_id[test_id]
                )
                same_ir = (
                    ir_reference_by_id[selected_id]
                    == ir_reference_by_id[test_id]
                )
                formal_equivalent = (
                    formal_group_by_id[selected_id]
                    == test_group
                )

                selection_detail_rows.append({
                    "fold": fold,
                    "id de teste": test_id,
                    "método de seleção": method,
                    "rótulo do método": (
                        ROTULOS_METODOS[method]
                    ),
                    "posição": rank,
                    "id selecionado": selected_id,
                    "similaridade": score,
                    "semente derivada": (
                        random_seed
                        if method == "random_seed42"
                        else None
                    ),
                    "tamanho do universo de treino": (
                        len(train_ids)
                    ),
                    "tamanho do universo elegível": (
                        len(eligible_ids)
                    ),
                    "ids formalmente excluídos": (
                        ", ".join(excluded_formal_ids)
                        if excluded_formal_ids
                        else "-"
                    ),
                    "grupo formal do teste": test_group,
                    "grupo formal selecionado": (
                        formal_group_by_id[selected_id]
                    ),
                    "pertence ao treino": (
                        selected_id in train_ids
                    ),
                    "vazamento do teste": (
                        selected_id == test_id
                    ),
                    "Nile idêntica à do teste": same_nile,
                    "IR idêntica à do teste": same_ir,
                    "equivalência formal com o teste": (
                        formal_equivalent
                    ),
                    "elegível": (
                        selected_id in eligible_ids
                    ),
                })

            for k in K_FEWSHOT:
                selected = top_five[:k]

                selection_jsonl_records.append({
                    "fold": fold,
                    "test_id": test_id,
                    "selection_method": method,
                    "selection_label": (
                        ROTULOS_METODOS[method]
                    ),
                    "k": k,
                    "selected_ids": [
                        item_id
                        for item_id, _ in selected
                    ],
                    "scores": [
                        (
                            None
                            if score is None
                            else float(score)
                        )
                        for _, score in selected
                    ],
                    "base_seed": SEED,
                    "derived_seed": (
                        random_seed
                        if method == "random_seed42"
                        else None
                    ),
                    "test_formal_group": test_group,
                    "excluded_formal_equivalent_ids": (
                        excluded_formal_ids
                    ),
                    "eligible_pool_size": len(eligible_ids),
                    "formal_exclusion_applied": True,
                })

selection_detail_df = pd.DataFrame(
    selection_detail_rows
)

formal_exclusions_by_test = {}

for item in selection_jsonl_records:
    key = (
        int(item["fold"]),
        str(item["test_id"]),
    )
    formal_exclusions_by_test[key] = tuple(
        item[
            "excluded_formal_equivalent_ids"
        ]
    )

FORMAL_EQUIVALENCE_INFO.update({
    "test_cases_with_exclusions": int(
        sum(
            bool(excluded_ids)
            for excluded_ids
            in formal_exclusions_by_test.values()
        )
    ),
    "excluded_train_test_pairs": int(
        sum(
            len(excluded_ids)
            for excluded_ids
            in formal_exclusions_by_test.values()
        )
    ),
})


# ----------------------------------------------------------
# 6.4 Auditoria das seleções e gravação
# ----------------------------------------------------------

expected_detail_rows = (
    TOTAL_EXEMPLOS
    * len(METODOS_FEWSHOT)
    * max(K_FEWSHOT)
)

if len(selection_detail_df) != expected_detail_rows:
    raise ValueError(
        "A seleção detalhada deve conter "
        f"{expected_detail_rows} linhas."
    )

if not selection_detail_df[
    "pertence ao treino"
].all():
    raise ValueError(
        "Uma demonstração foi selecionada fora "
        "do treino do fold."
    )

if not selection_detail_df["elegível"].all():
    raise ValueError(
        "Uma demonstração formalmente inelegível "
        "foi selecionada."
    )

if selection_detail_df[
    "vazamento do teste"
].any():
    raise ValueError(
        "O exemplo de teste foi incluído "
        "nas demonstrações."
    )

if selection_detail_df[
    "Nile idêntica à do teste"
].any():
    raise ValueError(
        "Uma Nile idêntica à referência do teste "
        "foi incluída nas demonstrações."
    )

if selection_detail_df[
    "IR idêntica à do teste"
].any():
    raise ValueError(
        "Uma IR idêntica à referência do teste "
        "foi incluída nas demonstrações."
    )

if selection_detail_df[
    "equivalência formal com o teste"
].any():
    raise ValueError(
        "Um exemplo formalmente equivalente ao teste "
        "foi incluído nas demonstrações."
    )

duplicated_demonstrations = (
    selection_detail_df.duplicated(
        subset=[
            "fold",
            "id de teste",
            "método de seleção",
            "id selecionado",
        ],
        keep=False,
    )
)

if duplicated_demonstrations.any():
    raise ValueError(
        "Uma ordenação few-shot contém "
        "demonstrações repetidas."
    )

expected_cumulative_rows = (
    TOTAL_EXEMPLOS
    * len(METODOS_FEWSHOT)
    * len(K_FEWSHOT)
)

if len(selection_jsonl_records) != (
    expected_cumulative_rows
):
    raise ValueError(
        "O mapa cumulativo deve conter "
        f"{expected_cumulative_rows} registros."
    )

selection_map = {
    (
        int(item["fold"]),
        item["test_id"],
        item["selection_method"],
        int(item["k"]),
    ): item
    for item in selection_jsonl_records
}

for fold in range(1, TOTAL_FOLDS + 1):
    test_ids = sorted(
        folds_df.loc[
            (folds_df["fold"] == fold)
            & (folds_df["split"] == "test"),
            "id",
        ].astype(str)
    )

    for test_id in test_ids:
        for method in METODOS_FEWSHOT:
            ids_1 = selection_map[
                (fold, test_id, method, 1)
            ]["selected_ids"]
            ids_3 = selection_map[
                (fold, test_id, method, 3)
            ]["selected_ids"]
            ids_5 = selection_map[
                (fold, test_id, method, 5)
            ]["selected_ids"]

            if (
                ids_3[:1] != ids_1
                or ids_5[:3] != ids_3
            ):
                raise ValueError(
                    "Seleção não cumulativa: "
                    f"{fold}, {test_id}, {method}"
                )

            for selected_id in ids_5:
                if (
                    formal_group_by_id[selected_id]
                    == formal_group_by_id[test_id]
                ):
                    raise ValueError(
                        "Equivalência formal detectada "
                        "no mapa cumulativo: "
                        f"{fold}, {test_id}, {method}, "
                        f"{selected_id}"
                    )

selection_detail_df.to_csv(
    F3_SELECTION_DETAIL_PATH,
    index=False,
    encoding="utf-8",
)

f3_common.write_jsonl(
    selection_jsonl_records,
    F3_SELECTION_JSONL_PATH,
)

selection_summary = (
    selection_detail_df.groupby(
        [
            "método de seleção",
            "rótulo do método",
        ],
        as_index=False,
    )
    .agg(
        exemplos_de_teste=(
            "id de teste",
            "nunique",
        ),
        seleções_registradas=(
            "id selecionado",
            "size",
        ),
        menor_universo_elegível=(
            "tamanho do universo elegível",
            "min",
        ),
        vazamentos_por_id=(
            "vazamento do teste",
            "sum",
        ),
        niles_idênticas=(
            "Nile idêntica à do teste",
            "sum",
        ),
        irs_idênticas=(
            "IR idêntica à do teste",
            "sum",
        ),
        equivalências_formais=(
            "equivalência formal com o teste",
            "sum",
        ),
    )
)

exibir_tabela(
    selection_summary,
    "Resumo da seleção few-shot",
    altura_px=320,
)

preview = selection_detail_df.head(15).copy()

exibir_tabela(
    preview,
    "Prévia das demonstrações selecionadas",
    altura_px=520,
)

SEMANTIC_SELECTION_INFO = {
    "model_id": MODELO_SEMANTICO_HF,
    "source_used": semantic_model_source,
    "local_source": semantic_local,
    "normalized_embeddings": True,
    "similarity": "cosine_via_dot_product",
}

del semantic_model
gc.collect()

try:
    import torch

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception:
    pass

print(
    "Seleções cumulativas k=1, k=3 e k=5: "
    "confirmadas"
)
print(
    "Exclusão de equivalências formais: confirmada"
)
print(
    "Vazamentos por ID, Nile, IR ou equivalência "
    "formal: 0"
)
print("Status: OK")

ID,grupo formal,tamanho do grupo,possui equivalente formal,ids formalmente equivalentes,SHA-256 da Nile de referência,SHA-256 da IR de referência,SHA-256 da assinatura formal
campi_023,formal_023,4,sim,"campi_043, campi_045, campi_050",558577168530903ee641a3dfbe04772323ba9869cfd481317b7d0f235c5d7b31,8f60a1ca9e922c24cc4890b217a8f80e7a880fea65216e43a77ca82ecf5680c4,80c0f1fa3e63c7074b20ef226eda160b8fbae5be359d5fb4b6042f6f3e53c363
campi_043,formal_023,4,sim,"campi_023, campi_045, campi_050",558577168530903ee641a3dfbe04772323ba9869cfd481317b7d0f235c5d7b31,8f60a1ca9e922c24cc4890b217a8f80e7a880fea65216e43a77ca82ecf5680c4,80c0f1fa3e63c7074b20ef226eda160b8fbae5be359d5fb4b6042f6f3e53c363
campi_045,formal_023,4,sim,"campi_023, campi_043, campi_050",558577168530903ee641a3dfbe04772323ba9869cfd481317b7d0f235c5d7b31,8f60a1ca9e922c24cc4890b217a8f80e7a880fea65216e43a77ca82ecf5680c4,80c0f1fa3e63c7074b20ef226eda160b8fbae5be359d5fb4b6042f6f3e53c363
campi_050,formal_023,4,sim,"campi_023, campi_043, campi_045",558577168530903ee641a3dfbe04772323ba9869cfd481317b7d0f235c5d7b31,8f60a1ca9e922c24cc4890b217a8f80e7a880fea65216e43a77ca82ecf5680c4,80c0f1fa3e63c7074b20ef226eda160b8fbae5be359d5fb4b6042f6f3e53c363


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

método de seleção,rótulo do método,exemplos de teste,seleções registradas,menor universo elegível,vazamentos por id,niles idênticas,irs idênticas,equivalências formais
lexical_tfidf,Lexical,50,250,37,0,0,0,0
random_seed42,Aleatória,50,250,37,0,0,0,0
semantic_embeddings,Semântica,50,250,37,0,0,0,0


fold,ID de teste,método de seleção,rótulo do método,posição,ID selecionado,similaridade,semente derivada,tamanho do universo de treino,tamanho do universo elegível,ids formalmente excluídos,grupo formal do teste,grupo formal selecionado,pertence ao treino,vazamento do teste,Nile idêntica à do teste,IR idêntica à do teste,equivalência formal com o teste,elegível
1,campi_001,random_seed42,Aleatória,1,campi_043,-,3773170263.0,40,40,-,formal_001,formal_023,sim,não,não,não,não,sim
1,campi_001,random_seed42,Aleatória,2,campi_048,-,3773170263.0,40,40,-,formal_001,formal_046,sim,não,não,não,não,sim
1,campi_001,random_seed42,Aleatória,3,campi_010,-,3773170263.0,40,40,-,formal_001,formal_010,sim,não,não,não,não,sim
1,campi_001,random_seed42,Aleatória,4,campi_034,-,3773170263.0,40,40,-,formal_001,formal_034,sim,não,não,não,não,sim
1,campi_001,random_seed42,Aleatória,5,campi_008,-,3773170263.0,40,40,-,formal_001,formal_008,sim,não,não,não,não,sim
1,campi_001,lexical_tfidf,Lexical,1,campi_030,0.200435,-,40,40,-,formal_001,formal_030,sim,não,não,não,não,sim
1,campi_001,lexical_tfidf,Lexical,2,campi_031,0.148141,-,40,40,-,formal_001,formal_031,sim,não,não,não,não,sim
1,campi_001,lexical_tfidf,Lexical,3,campi_026,0.140009,-,40,40,-,formal_001,formal_026,sim,não,não,não,não,sim
1,campi_001,lexical_tfidf,Lexical,4,campi_032,0.133384,-,40,40,-,formal_001,formal_032,sim,não,não,não,não,sim
1,campi_001,lexical_tfidf,Lexical,5,campi_045,0.132498,-,40,40,-,formal_001,formal_023,sim,não,não,não,não,sim


Seleções cumulativas k=1, k=3 e k=5: confirmadas
Exclusão de equivalências formais: confirmada
Vazamentos por ID, Nile, IR ou equivalência formal: 0
Status: OK


## Bloco 7 - Montagem padronizada e auditoria completa dos prompts

### Objetivo

Este bloco transforma as seleções do Bloco 6 em demonstrações e prompts. A montagem é centralizada para que todas as estratégias utilizem a mesma formatação e as mesmas barreiras contra vazamento.

### Conteúdo das demonstrações

Somente exemplos elegíveis do split de treino podem fornecer:

- intenção em linguagem natural;
- Nile de referência;
- IR de referência;
- R1, R2 ou R3, conforme a estratégia.

Os formatos utilizados são:

- F3-A e F3-B: `NL → Nile`;
- F3-C: `NL → IR` e `IR → Nile`;
- F3-D: `NL + R3 → Nile`;
- F3-E: `NL + R1 → IR` e `IR + R2 → Nile`.

Nas estratégias em duas etapas, a segunda etapa recebe uma saída gerada em tempo de execução. A IR de referência do exemplo de teste não é utilizada.

### Proteções verificadas

Para cada prompt, o bloco confere que não aparecem:

- a Nile de referência do teste;
- a IR de referência do teste;
- R1, R2 ou R3 do teste;
- demonstrações formalmente equivalentes ao teste;
- placeholders não resolvidos.

### Escopo da auditoria

A auditoria não utiliza apenas um exemplo ilustrativo. Ela cobre:

- os 50 exemplos de teste;
- as condições zero-shot de F3-A e F3-B;
- os três métodos few-shot;
- `k=1`, `k=3` e `k=5`;
- as etapas diretas, intermediárias e de autocorreção das estratégias.

O conteúdo integral dos prompts não é gravado nesse artefato. São registrados apenas os metadados necessários à auditoria, como fold, ID de teste, método, `k`, etapa, demonstrações, tamanho e indicadores de vazamento.

### Saídas

O bloco gera:

- `f3_prompt_templates.json`, com os metadados dos templates;
- `f3_prompt_audit.csv`, com a auditoria completa da montagem.

Qualquer referência de teste, equivalência formal ou placeholder pendente interrompe a execução antes do empacotamento.

In [7]:
# ----------------------------------------------------------
# 7.1 Gravação dos metadados dos templates
# ----------------------------------------------------------

PROMPT_METADATA_EXPORT = {
    "dataset": DATASET_ID,
    "model": MODELO_ALUNO,
    "templates": prompt_metadata,
}

f3_common.write_json(
    PROMPT_METADATA_EXPORT,
    F3_PROMPT_METADATA_PATH,
)


# ----------------------------------------------------------
# 7.2 Recuperação cumulativa das demonstrações
# ----------------------------------------------------------


def selected_ids_for(
    fold,
    test_id,
    method,
    k,
):
    if int(k) == 0:
        return []

    key = (
        int(fold),
        str(test_id),
        str(method),
        int(k),
    )

    if key not in selection_map:
        raise KeyError(
            f"Seleção ausente: {key}"
        )

    return list(
        selection_map[key]["selected_ids"]
    )


# ----------------------------------------------------------
# 7.3 Montagem comum dos prompts de auditoria
# ----------------------------------------------------------


def montar_prompts_auditoria(
    test_record,
    selected_ids,
):
    demos_a = f3_common.build_demos(
        selected_ids,
        records_by_id,
        "A",
    )
    demos_c_ir = f3_common.build_demos(
        selected_ids,
        records_by_id,
        "C_IR",
    )
    demos_c_nile = f3_common.build_demos(
        selected_ids,
        records_by_id,
        "C_IR_NILE",
    )
    demos_d = f3_common.build_demos(
        selected_ids,
        records_by_id,
        "D_R3",
    )
    demos_e_ir = f3_common.build_demos(
        selected_ids,
        records_by_id,
        "E_R1",
    )
    demos_e_nile = f3_common.build_demos(
        selected_ids,
        records_by_id,
        "E_R2",
    )

    runtime_ir = {
        "generated": "placeholder_for_runtime"
    }
    runtime_nile = (
        "generated_nile_placeholder"
    )
    runtime_feedback = (
        "validator_feedback_placeholder"
    )

    prompts = {
        "F3-A - tradução direta": (
            f3_common.build_prompt_a(
                prompt_templates[
                    "f3a_traducao_direta"
                ],
                test_record["nl"],
                demos_a,
            )
        ),
        "F3-B - autocorreção": (
            f3_common.build_prompt_autocorrection(
                prompt_templates[
                    "f3_autocorrecao_nile"
                ],
                test_record["nl"],
                "Nenhum.",
                runtime_nile,
                runtime_feedback,
                demos_a,
            )
        ),
        "F3-C - geração da IR": (
            f3_common.build_prompt_c_ir(
                prompt_templates[
                    "f3c_gerar_ir_teste"
                ],
                test_record["nl"],
                demos_c_ir,
                ir_schema,
            )
        ),
        "F3-C - geração da Nile": (
            f3_common.build_prompt_c_nile(
                prompt_templates[
                    "f3c_gerar_nile_a_partir_ir"
                ],
                runtime_ir,
                demos_c_nile,
            )
        ),
        "F3-C - autocorreção": (
            f3_common.build_prompt_autocorrection(
                prompt_templates[
                    "f3_autocorrecao_nile"
                ],
                test_record["nl"],
                f3_common.canonical_json(
                    runtime_ir
                ),
                runtime_nile,
                runtime_feedback,
                demos_c_nile,
            )
        ),
        "F3-D - tradução com R3": (
            f3_common.build_prompt_d(
                prompt_templates[
                    "f3d_traducao_com_r3"
                ],
                test_record["nl"],
                demos_d,
            )
        ),
        "F3-D - autocorreção": (
            f3_common.build_prompt_autocorrection(
                prompt_templates[
                    "f3_autocorrecao_nile"
                ],
                test_record["nl"],
                "Nenhum.",
                runtime_nile,
                runtime_feedback,
                demos_d,
            )
        ),
        "F3-E - geração da IR com R1": (
            f3_common.build_prompt_e_ir(
                prompt_templates[
                    "f3e_gerar_ir_com_r1"
                ],
                test_record["nl"],
                demos_e_ir,
                ir_schema,
            )
        ),
        "F3-E - geração da Nile com R2": (
            f3_common.build_prompt_e_nile(
                prompt_templates[
                    "f3e_gerar_nile_com_r2"
                ],
                runtime_ir,
                demos_e_nile,
            )
        ),
        "F3-E - autocorreção": (
            f3_common.build_prompt_autocorrection(
                prompt_templates[
                    "f3_autocorrecao_nile"
                ],
                test_record["nl"],
                f3_common.canonical_json(
                    runtime_ir
                ),
                runtime_nile,
                runtime_feedback,
                demos_e_nile,
            )
        ),
    }

    return prompts


# ----------------------------------------------------------
# 7.4 Auditoria completa dos prompts zero-shot
# ----------------------------------------------------------

prompt_audit_rows = []

for fold in range(1, TOTAL_FOLDS + 1):
    test_ids = sorted(
        folds_df.loc[
            (folds_df["fold"] == fold)
            & (folds_df["split"] == "test"),
            "id",
        ].astype(str)
    )

    for test_id in test_ids:
        test_record = records_by_id[test_id]

        prompts_zero_shot = {
            "F3-A - tradução direta zero-shot": (
                f3_common.build_prompt_a(
                    prompt_templates[
                        "f3a_traducao_direta"
                    ],
                    test_record["nl"],
                    "",
                )
            ),
            "F3-B - autocorreção zero-shot": (
                f3_common.build_prompt_autocorrection(
                    prompt_templates[
                        "f3_autocorrecao_nile"
                    ],
                    test_record["nl"],
                    "Nenhum.",
                    "generated_nile_placeholder",
                    "validator_feedback_placeholder",
                    "",
                )
            ),
        }

        test_nile = test_record["nile"]
        canonical_test_ir = (
            f3_common.canonical_json(
                test_record["ir"]
            )
        )

        for stage, prompt in (
            prompts_zero_shot.items()
        ):
            placeholders = re.findall(
                r"\{\{[A-Z0-9_]+\}\}",
                prompt,
            )

            leakage_flags = {
                "Nile de referência no prompt": (
                    test_nile in prompt
                ),
                "IR de referência no prompt": (
                    canonical_test_ir in prompt
                ),
                "R1 de referência no prompt": (
                    test_record["r1"] in prompt
                ),
                "R2 de referência no prompt": (
                    test_record["r2"] in prompt
                ),
                "R3 de referência no prompt": (
                    test_record["r3"] in prompt
                ),
                "equivalência formal nas demonstrações": (
                    False
                ),
            }

            leakage_detected = any(
                leakage_flags.values()
            )

            prompt_audit_rows.append({
                "fold": fold,
                "id de teste": test_id,
                "método de seleção": METODO_ZERO_SHOT,
                "k": 0,
                "etapa": stage,
                "demonstrações": "-",
                "caracteres do prompt": len(prompt),
                "placeholders pendentes": (
                    len(placeholders)
                ),
                **leakage_flags,
                "vazamento detectado": (
                    leakage_detected
                ),
                "status": (
                    "ERRO"
                    if leakage_detected or placeholders
                    else "OK"
                ),
            })


# ----------------------------------------------------------
# 7.5 Auditoria completa dos prompts few-shot
# ----------------------------------------------------------

for selection_record in selection_jsonl_records:
    fold = int(selection_record["fold"])
    test_id = str(
        selection_record["test_id"]
    )
    method = str(
        selection_record["selection_method"]
    )
    k = int(selection_record["k"])
    selected_ids = list(
        selection_record["selected_ids"]
    )

    test_record = records_by_id[test_id]
    test_group = formal_group_by_id[test_id]

    equivalent_selected_ids = [
        selected_id
        for selected_id in selected_ids
        if formal_group_by_id[selected_id] == test_group
    ]

    prompts = montar_prompts_auditoria(
        test_record,
        selected_ids,
    )

    test_nile = test_record["nile"]
    canonical_test_ir = (
        f3_common.canonical_json(
            test_record["ir"]
        )
    )

    for stage, prompt in prompts.items():
        placeholders = re.findall(
            r"\{\{[A-Z0-9_]+\}\}",
            prompt,
        )

        leakage_flags = {
            "Nile de referência no prompt": (
                test_nile in prompt
            ),
            "IR de referência no prompt": (
                canonical_test_ir in prompt
            ),
            "R1 de referência no prompt": (
                test_record["r1"] in prompt
            ),
            "R2 de referência no prompt": (
                test_record["r2"] in prompt
            ),
            "R3 de referência no prompt": (
                test_record["r3"] in prompt
            ),
            "equivalência formal nas demonstrações": (
                bool(equivalent_selected_ids)
            ),
        }

        leakage_detected = any(
            leakage_flags.values()
        )

        prompt_audit_rows.append({
            "fold": fold,
            "id de teste": test_id,
            "método de seleção": method,
            "k": k,
            "etapa": stage,
            "demonstrações": (
                ", ".join(selected_ids)
            ),
            "caracteres do prompt": len(prompt),
            "placeholders pendentes": (
                len(placeholders)
            ),
            **leakage_flags,
            "vazamento detectado": (
                leakage_detected
            ),
            "status": (
                "ERRO"
                if leakage_detected or placeholders
                else "OK"
            ),
        })

prompt_audit_df = pd.DataFrame(
    prompt_audit_rows
)


# ----------------------------------------------------------
# 7.6 Validação e gravação da auditoria
# ----------------------------------------------------------

expected_zero_shot_prompts = (
    TOTAL_EXEMPLOS * 2
)

expected_few_shot_prompts = (
    len(selection_jsonl_records) * 10
)

expected_prompt_audit_rows = (
    expected_zero_shot_prompts
    + expected_few_shot_prompts
)

if len(prompt_audit_df) != (
    expected_prompt_audit_rows
):
    raise ValueError(
        "Quantidade inesperada de prompts auditados: "
        f"{len(prompt_audit_df)}. "
        f"Esperado: {expected_prompt_audit_rows}."
    )

if prompt_audit_df[
    "id de teste"
].nunique() != TOTAL_EXEMPLOS:
    raise ValueError(
        "A auditoria não cobriu os 50 exemplos "
        "de teste."
    )

if (
    prompt_audit_df[
        "placeholders pendentes"
    ] > 0
).any():
    raise ValueError(
        "Um ou mais prompts mantêm placeholders "
        "não resolvidos."
    )

if prompt_audit_df[
    "vazamento detectado"
].any():
    failures = prompt_audit_df.loc[
        prompt_audit_df[
            "vazamento detectado"
        ],
        [
            "fold",
            "id de teste",
            "método de seleção",
            "k",
            "etapa",
        ],
    ]

    raise ValueError(
        "Vazamento detectado na auditoria "
        "completa dos prompts:\n"
        + failures.head(20).to_string(
            index=False
        )
    )

prompt_audit_df.to_csv(
    F3_PROMPT_AUDIT_PATH,
    index=False,
    encoding="utf-8",
)

prompt_audit_summary = (
    prompt_audit_df.groupby(
        ["etapa"],
        as_index=False,
    )
    .agg(
        prompts_auditados=(
            "id de teste",
            "size",
        ),
        exemplos_de_teste=(
            "id de teste",
            "nunique",
        ),
        placeholders_pendentes=(
            "placeholders pendentes",
            "sum",
        ),
        vazamentos=(
            "vazamento detectado",
            "sum",
        ),
    )
)

exibir_tabela(
    prompt_audit_summary,
    "Auditoria completa da montagem dos prompts",
    altura_px=520,
)

prompt_audit_preview = (
    prompt_audit_df.head(15).copy()
)

exibir_tabela(
    prompt_audit_preview,
    "Prévia da auditoria dos prompts",
    altura_px=520,
)

PROMPT_AUDIT_INFO = {
    "records": len(prompt_audit_df),
    "zero_shot_records": (
        expected_zero_shot_prompts
    ),
    "few_shot_records": (
        expected_few_shot_prompts
    ),
    "test_examples": int(
        prompt_audit_df[
            "id de teste"
        ].nunique()
    ),
    "stages": sorted(
        prompt_audit_df["etapa"].unique()
    ),
    "pending_placeholders": int(
        prompt_audit_df[
            "placeholders pendentes"
        ].sum()
    ),
    "leakage_records": int(
        prompt_audit_df[
            "vazamento detectado"
        ].sum()
    ),
    "formal_equivalence_leakage_records": int(
        prompt_audit_df[
            "equivalência formal nas demonstrações"
        ].sum()
    ),
}

print(
    f"Prompts auditados: {len(prompt_audit_df)}"
)
print(
    "Exemplos de teste cobertos: "
    f"{PROMPT_AUDIT_INFO['test_examples']}"
)
print(
    "Vazamentos por referência ou equivalência "
    "formal: 0"
)
print("Status: OK")

etapa,prompts auditados,exemplos de teste,placeholders pendentes,vazamentos
F3-A - tradução direta,450,50,0,0
F3-A - tradução direta zero-shot,50,50,0,0
F3-B - autocorreção,450,50,0,0
F3-B - autocorreção zero-shot,50,50,0,0
F3-C - autocorreção,450,50,0,0
F3-C - geração da IR,450,50,0,0
F3-C - geração da Nile,450,50,0,0
F3-D - autocorreção,450,50,0,0
F3-D - tradução com R3,450,50,0,0
F3-E - autocorreção,450,50,0,0


fold,ID de teste,método de seleção,k,etapa,demonstrações,caracteres do prompt,placeholders pendentes,Nile de referência no prompt,IR de referência no prompt,R1 de referência no prompt,R2 de referência no prompt,R3 de referência no prompt,equivalência formal nas demonstrações,vazamento detectado,status
1,campi_001,zero_shot,0,F3-A - tradução direta zero-shot,-,1923,0,não,não,não,não,não,não,não,OK
1,campi_001,zero_shot,0,F3-B - autocorreção zero-shot,-,2260,0,não,não,não,não,não,não,não,OK
1,campi_002,zero_shot,0,F3-A - tradução direta zero-shot,-,1748,0,não,não,não,não,não,não,não,OK
1,campi_002,zero_shot,0,F3-B - autocorreção zero-shot,-,2085,0,não,não,não,não,não,não,não,OK
1,campi_011,zero_shot,0,F3-A - tradução direta zero-shot,-,1835,0,não,não,não,não,não,não,não,OK
1,campi_011,zero_shot,0,F3-B - autocorreção zero-shot,-,2172,0,não,não,não,não,não,não,não,OK
1,campi_012,zero_shot,0,F3-A - tradução direta zero-shot,-,1728,0,não,não,não,não,não,não,não,OK
1,campi_012,zero_shot,0,F3-B - autocorreção zero-shot,-,2065,0,não,não,não,não,não,não,não,OK
1,campi_020,zero_shot,0,F3-A - tradução direta zero-shot,-,1740,0,não,não,não,não,não,não,não,OK
1,campi_020,zero_shot,0,F3-B - autocorreção zero-shot,-,2077,0,não,não,não,não,não,não,não,OK


Prompts auditados: 4600
Exemplos de teste cobertos: 50
Vazamentos por referência ou equivalência formal: 0
Status: OK


## Bloco 8 - Pós-processamento, validação e métricas

### Objetivo

Este bloco fixa o tratamento aplicado às respostas brutas do modelo aluno. O mesmo procedimento será utilizado em todas as estratégias para evitar que diferenças de limpeza, extração ou avaliação alterem artificialmente os resultados.

### Extração mecânica

As funções aceitam respostas que possam conter:

- texto puro;
- blocos Markdown;
- cercas de código para Nile;
- cercas de código JSON para IR.

A extração remove apenas elementos externos previsíveis e recupera o conteúdo candidato. Ela não consulta a referência, não completa campos ausentes, não corrige sintaxe e não reescreve a saída do modelo.

### Validação da Nile

Depois da extração, a saída Nile é submetida ao mesmo validador da F0. São registrados:

- validade sintática;
- feedback do parser;
- expressão efetivamente avaliada.

A referência Nile só é acessada após essa etapa, exclusivamente para o cálculo das métricas.

### Validação da IR

Nas estratégias F3-C e F3-E, a IR gerada é:

1. extraída da resposta;
2. interpretada como JSON;
3. validada contra o esquema da F1;
4. submetida às verificações determinísticas de `ir_core.py`.

Uma IR inválida não é substituída pela referência.

### Métricas

O padrão comum contém:

- **PSR**, que indica aceitação sintática;
- **EM**, correspondência textual exata;
- **ED**, distância de edição;
- **NED**, distância de edição normalizada;
- **SLA-S**, aderência estrutural binária;
- **SLA-F**, aderência complementar por campos.

PSR mede somente sintaxe. SLA-S e SLA-F analisam a estrutura extraída, conforme as regras definidas na F0.

### Testes determinísticos

O bloco testa:

- extração de Nile cercada por Markdown;
- rejeição de uma expressão Nile inválida;
- extração de IR cercada por Markdown;
- validação de um registro completo pelo esquema JSON;
- identidade entre uma referência e ela mesma.

### Resultado esperado

Ao final, o notebook confirma que o pós-processamento é independente da referência e que o esquema comum aceita um registro completo. Essas funções são exportadas para os notebooks das estratégias.

In [8]:
# ----------------------------------------------------------
# 8.1 Testes da extração de Nile e das métricas
# ----------------------------------------------------------

sample_id = sorted(records_by_id)[0]
sample_nile = records_by_id[sample_id]["nile"]
sample_ir = records_by_id[sample_id]["ir"]

wrapped_nile = f"```nile\n{sample_nile}\n```"
nile_evaluation = f3_common.evaluate_nile_output(
    sample_nile,
    wrapped_nile,
    validator,
    nile_metrics,
)

invalid_evaluation = f3_common.evaluate_nile_output(
    sample_nile,
    "define intent broken:",
    validator,
    nile_metrics,
)

wrapped_ir = "```json\n" + json.dumps(
    sample_ir,
    ensure_ascii=False,
    indent=2,
) + "\n```"
ir_evaluation = f3_common.validate_generated_ir(
    wrapped_ir,
    ir_schema,
    ir_core,
)

if nile_evaluation["nile"] != sample_nile:
    raise ValueError("A extração mecânica alterou a referência Nile.")

if not (
    nile_evaluation["psr"] == 1
    and nile_evaluation["em"] == 1
    and nile_evaluation["ed"] == 0
    and nile_evaluation["ned"] == 0
    and nile_evaluation["sla_s"] == 1
    and nile_evaluation["sla_f"] == 1
):
    raise ValueError("A avaliação da saída Nile válida falhou.")

if invalid_evaluation["psr"] != 0:
    raise ValueError("A saída Nile inválida não foi rejeitada.")

if not ir_evaluation["ir_valid"]:
    raise ValueError("A IR válida não foi recuperada corretamente.")


# ----------------------------------------------------------
# 8.2 Teste do padrão de registro e do esquema JSON
# ----------------------------------------------------------

from jsonschema import Draft202012Validator

sample_result = f3_common.make_result_record(
    strategy="F3-A",
    variant="sem_ac",
    fold=1,
    id=sample_id,
    k=0,
    selection_method=METODO_ZERO_SHOT,
    selection_label=ROTULOS_METODOS[METODO_ZERO_SHOT],
    nl=records_by_id[sample_id]["nl"],
    reference_nile=sample_nile,
    prompt_primary="prompt de teste",
    response_primary_raw=wrapped_nile,
    nile_initial=sample_nile,
    nile_evaluated=sample_nile,
    syntax_valid=True,
    validator_feedback=nile_evaluation["validator_feedback"],
    psr=nile_evaluation["psr"],
    em=nile_evaluation["em"],
    ed=nile_evaluation["ed"],
    ned=nile_evaluation["ned"],
    sla_s=nile_evaluation["sla_s"],
    sla_f=nile_evaluation["sla_f"],
    status="completed",
)

schema_errors = list(
    Draft202012Validator(RESULT_SCHEMA).iter_errors(sample_result)
)

if schema_errors:
    raise ValueError(
        "O registro comum não atende ao esquema: "
        + " | ".join(error.message for error in schema_errors)
    )

test_rows = [
    {"teste": "Nile cercada por Markdown", "resultado esperado": "extração exata", "resultado": nile_evaluation["em"], "status": "OK"},
    {"teste": "Nile inválida", "resultado esperado": "PSR = 0", "resultado": invalid_evaluation["psr"], "status": "OK"},
    {"teste": "IR cercada por Markdown", "resultado esperado": "IR válida", "resultado": ir_evaluation["ir_valid"], "status": "OK"},
    {"teste": "Registro detalhado", "resultado esperado": "esquema válido", "resultado": len(schema_errors) == 0, "status": "OK"},
]

exibir_tabela(
    pd.DataFrame(test_rows),
    "Testes do pós-processamento e da avaliação",
    altura_px=320,
)

metric_pattern_df = pd.DataFrame([
    {"métrica": "PSR", "campo interno": "psr", "melhor valor": 1, "observação": "validade sintática"},
    {"métrica": "EM", "campo interno": "em", "melhor valor": 1, "observação": "correspondência textual exata"},
    {"métrica": "ED", "campo interno": "ed", "melhor valor": 0, "observação": "distância de edição"},
    {"métrica": "NED", "campo interno": "ned", "melhor valor": 0, "observação": "distância de edição normalizada"},
    {"métrica": "SLA-S", "campo interno": "sla_s", "melhor valor": 1, "observação": "aderência estrutural binária"},
    {"métrica": "SLA-F", "campo interno": "sla_f", "melhor valor": 1, "observação": "aderência por campos"},
])

exibir_tabela(
    metric_pattern_df,
    "Padrão comum das métricas",
    altura_px=340,
)

print("Pós-processamento independente da referência: confirmado")
print("Status: OK")

teste,resultado esperado,resultado,status
Nile cercada por Markdown,extração exata,1,OK
Nile inválida,PSR = 0,0,OK
IR cercada por Markdown,IR válida,sim,OK
Registro detalhado,esquema válido,sim,OK


métrica,campo interno,melhor valor,observação
PSR,psr,1,validade sintática
EM,em,1,correspondência textual exata
ED,ed,0,distância de edição
NED,ned,0,distância de edição normalizada
SLA-S,sla_s,1,aderência estrutural binária
SLA-F,sla_f,1,aderência por campos


Pós-processamento independente da referência: confirmado
Status: OK


## Bloco 9 - Modelo aluno e geração determinística

### Objetivo

Este bloco prepara o acesso ao Qwen2.5-1.5B-Instruct e fixa as condições de inferência que serão usadas nas estratégias. Como a F3-modelo não executa nenhuma condição experimental, o carregamento efetivo do modelo é opcional.

### Resolução da fonte do modelo

O notebook procura primeiro uma cópia local do modelo em `/kaggle/input`. Essa é a opção preferencial para execuções reprodutíveis e sem dependência de rede. Quando o modelo não é encontrado localmente, a configuração pode permitir o uso da fonte registrada no Hugging Face.

A execução é interrompida quando:

- o modelo não está disponível localmente;
- o download não foi autorizado;
- não existe uma fonte utilizável.

### Configuração de inferência

São registrados:

- identificador do modelo;
- origem efetivamente resolvida;
- disponibilidade de GPU;
- ausência de amostragem;
- semente global;
- limite de novos tokens para Nile;
- limite de novos tokens para IR;
- versões das bibliotecas principais.

A geração utiliza a mesma função determinística em todas as estratégias. O texto de entrada é processado com o tokenizer correspondente ao modelo, e somente os tokens novos são devolvidos como resposta.

### Carregamento opcional

Duas chaves controlam este notebook:

- uma permite carregar o modelo apenas para verificar a compatibilidade do ambiente;
- outra permite executar um teste curto de geração.

Por padrão, essas opções podem permanecer desativadas para economizar memória e tempo. O carregamento completo será realizado nos notebooks das estratégias.

### Gestão de recursos

Quando o teste opcional é executado, o modelo e o tokenizer são removidos da memória ao final. A memória da GPU é liberada antes do avanço para o empacotamento.

### Saída

As informações do ambiente e da fonte do modelo são incorporadas à configuração final. Os pesos do Qwen não são copiados para o pacote operacional.

In [9]:
# ----------------------------------------------------------
# 9.1 Resolução da fonte do modelo aluno
# ----------------------------------------------------------

local_student_dir = f3_common.resolve_local_model_dir(
    KAGGLE_INPUT_DIR,
    ["qwen2.5", "1.5b", "instruct"],
)

if local_student_dir is not None:
    student_model_source = str(local_student_dir)
    student_model_local = True
else:
    student_model_source = MODELO_ALUNO_HF
    student_model_local = False
    if not PERMITIR_DOWNLOAD_MODELOS:
        raise FileNotFoundError(
            "O Qwen2.5-1.5B-Instruct não foi encontrado em /kaggle/input. "
            "Adicione o modelo como dataset ou permita download."
        )


# ----------------------------------------------------------
# 9.2 Informações do ambiente de execução
# ----------------------------------------------------------

import torch

package_versions = {}
for package_name in [
    "transformers",
    "sentence-transformers",
    "accelerate",
    "torch",
    "pandas",
    "numpy",
    "scikit-learn",
    "lark",
    "jsonschema",
]:
    try:
        package_versions[package_name] = importlib.metadata.version(
            package_name
        )
    except importlib.metadata.PackageNotFoundError:
        package_versions[package_name] = None

MODEL_RUNTIME_INFO = {
    "model_name": MODELO_ALUNO,
    "model_id": MODELO_ALUNO_HF,
    "source_used": student_model_source,
    "local_source": student_model_local,
    "do_sample": GERACAO["do_sample"],
    "seed": GERACAO["seed"],
    "max_new_tokens_nile": GERACAO["max_new_tokens_nile"],
    "max_new_tokens_ir": GERACAO["max_new_tokens_ir"],
    "cuda_available": torch.cuda.is_available(),
    "cuda_device": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else None
    ),
    "loaded_in_f3_modelo": False,
    "smoke_test_executed": False,
}


# ----------------------------------------------------------
# 9.3 Carregamento opcional e teste curto
# ----------------------------------------------------------

if CARREGAR_MODELO_ALUNO_NA_F3_MODELO:
    tokenizer, student_model = f3_common.load_student_model(
        student_model_source,
        use_cuda=True,
        local_files_only=student_model_local,
    )
    MODEL_RUNTIME_INFO["loaded_in_f3_modelo"] = True

    if EXECUTAR_TESTE_CURTO_MODELO_ALUNO:
        smoke = f3_common.generate_student_output(
            "Responda apenas com a palavra OK.",
            tokenizer,
            student_model,
            max_new_tokens=8,
            seed=SEED,
        )
        MODEL_RUNTIME_INFO["smoke_test_executed"] = True
        MODEL_RUNTIME_INFO["smoke_test_generated_tokens"] = smoke[
            "generated_tokens"
        ]

    del student_model
    del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

model_status_df = pd.DataFrame([
    {
        "item": "Modelo aluno",
        "valor": MODELO_ALUNO,
        "status": "configurado",
    },
    {
        "item": "Fonte resolvida",
        "valor": student_model_source,
        "status": "local" if student_model_local else "Hugging Face",
    },
    {
        "item": "Amostragem",
        "valor": str(GERACAO["do_sample"]),
        "status": "determinística",
    },
    {
        "item": "Máximo de tokens Nile",
        "valor": GERACAO["max_new_tokens_nile"],
        "status": "fixo",
    },
    {
        "item": "Máximo de tokens IR",
        "valor": GERACAO["max_new_tokens_ir"],
        "status": "fixo",
    },
    {
        "item": "GPU",
        "valor": MODEL_RUNTIME_INFO["cuda_device"] or "não detectada",
        "status": "disponível" if torch.cuda.is_available() else "CPU",
    },
])

exibir_tabela(
    model_status_df,
    "Configuração do modelo aluno",
    altura_px=360,
)

print("Funções de carregamento e geração prontas para as estratégias")
print("Status: OK")

item,valor,status
Modelo aluno,Qwen2.5-1.5B-Instruct,configurado
Fonte resolvida,Qwen/Qwen2.5-1.5B-Instruct,Hugging Face
Amostragem,não,determinística
Máximo de tokens Nile,256,fixo
Máximo de tokens IR,512,fixo
GPU,Tesla T4,disponível


Funções de carregamento e geração prontas para as estratégias
Status: OK


## Bloco 10 - Manifesto e empacotamento final

### Objetivo

Este bloco encerra a preparação da base comum da Fase 3. Ele reúne os artefatos produzidos nos blocos anteriores, registra sua proveniência e cria o pacote que será consumido pelos notebooks F3-A, F3-B, F3-C, F3-D e F3-E.

### Configuração consolidada

O arquivo `f3_config.json` registra:

- conjunto experimental, folds e semente;
- modelo aluno e fonte resolvida;
- métodos de seleção e valores de `k`;
- regra cumulativa;
- exclusão de equivalências formais;
- parâmetros de geração;
- estratégias e variantes;
- métricas comuns;
- cobertura da auditoria dos prompts;
- barreiras contra vazamento.

### Conteúdo do pacote

O pacote contém:

- módulo comum;
- configuração, grade e esquema dos resultados;
- auditoria de F0, F1 e F2;
- grupos de equivalência formal;
- seleção few-shot detalhada e cumulativa;
- metadados dos templates;
- auditoria completa dos prompts;
- cópias dos artefatos necessários de F0, F1 e F2;
- templates das estratégias e da autocorreção;
- manifesto final.

Os pesos dos modelos e os resultados experimentais não são incluídos.

### Manifesto e integridade

Para cada arquivo são registrados caminho relativo, tamanho e SHA-256. O manifesto também informa:

- os hashes das fases anteriores;
- os grupos formais duplicados;
- o número de exclusões formais;
- o total de prompts auditados;
- a ausência de vazamentos por ID, Nile, IR, justificativa ou equivalência formal;
- os totais esperados da execução.

### Verificações finais

Antes de concluir, o bloco confirma que:

- os 30 arquivos esperados estão presentes;
- a grade contém 74 condições;
- o total principal permanece em 3.700 registros;
- a seleção utiliza somente candidatos elegíveis;
- nenhum equivalente formal foi selecionado;
- todos os prompts previstos foram auditados;
- o ZIP pode ser aberto e corresponde à estrutura registrada.

### Resultado

O artefato produzido é:

```text
/kaggle/working/f3_modelo_operacional.zip
```

Os notebooks das estratégias deverão consumir esse pacote sem modificar a grade, as seleções, os templates ou as funções comuns.

In [10]:
# ----------------------------------------------------------
# 10.1 Configuração consolidada da F3-modelo
# ----------------------------------------------------------

F3_CONFIG = {
    "phase": FASE,
    "dataset": DATASET_ID,
    "examples": TOTAL_EXEMPLOS,
    "folds": {
        "count": TOTAL_FOLDS,
        "train_per_fold": EXEMPLOS_TREINO_POR_FOLD,
        "test_per_fold": EXEMPLOS_TESTE_POR_FOLD,
        "seed": SEED,
    },
    "student_model": MODEL_RUNTIME_INFO,
    "semantic_selection": SEMANTIC_SELECTION_INFO,
    "few_shot": {
        "k_values": K_FEWSHOT,
        "methods": METODOS_FEWSHOT,
        "cumulative": True,
        "train_only": True,
        "formal_equivalence_exclusion": (
            FORMAL_EQUIVALENCE_INFO
        ),
        "lexical": {
            "representation": "TF-IDF",
            "ngram_range": [1, 2],
            "similarity": "cosine",
            "fit_scope": (
                "train split of each fold"
            ),
        },
    },
    "prompt_audit": PROMPT_AUDIT_INFO,
    "generation": GERACAO,
    "strategies": STRATEGY_PLAN,
    "expected": {
        "main_configurations": (
            TOTAL_CONFIGURACOES_PRINCIPAIS
        ),
        "main_records": (
            TOTAL_REGISTROS_PRINCIPAIS
        ),
        "final_autocorrection_maximum_eligible_records": (
            TOTAL_REGISTROS_AC_FINAL_MAXIMO
        ),
    },
    "metrics": [
        "psr",
        "em",
        "ed",
        "ned",
        "sla_s",
        "sla_f",
    ],
    "leakage_guards": {
        "test_example_never_used_as_demonstration": True,
        "same_reference_nile_demonstration_excluded": True,
        "same_reference_ir_demonstration_excluded": True,
        "formal_equivalent_demonstration_excluded": True,
        "test_reference_nile_not_in_prompt": True,
        "test_reference_ir_not_in_prompt": True,
        "test_rationales_not_in_prompt": True,
        "prompt_audit_covers_all_test_examples": True,
        "prompt_audit_leakage_records": int(
            PROMPT_AUDIT_INFO[
                "leakage_records"
            ]
        ),
    },
}
f3_common.write_json(F3_CONFIG, F3_CONFIG_PATH)


# ----------------------------------------------------------
# 10.2 Preparação dos arquivos do pacote
# ----------------------------------------------------------

if PACOTE_DIR.exists():
    shutil.rmtree(PACOTE_DIR)
PACOTE_DIR.mkdir(parents=True, exist_ok=True)

files_to_package = []


def add_package_file(source, destination):
    source = Path(source)
    if not source.is_file():
        raise FileNotFoundError(source)
    destination_path = PACOTE_DIR / destination
    destination_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(source, destination_path)
    files_to_package.append(destination_path)


add_package_file(F3_COMMON_PATH, "f3_common.py")
add_package_file(F3_CONFIG_PATH, "f3_config.json")
add_package_file(F3_GRID_PATH, "f3_grid.csv")
add_package_file(F3_INPUT_AUDIT_PATH, "f3_input_audit.csv")
add_package_file(F3_SELECTION_DETAIL_PATH, "f3_selecao_fewshot.csv")
add_package_file(F3_SELECTION_JSONL_PATH, "f3_exemplos_fewshot.jsonl")
add_package_file(F3_FORMAL_EQUIVALENCE_PATH, "f3_equivalencia_formal.csv")
add_package_file(F3_PROMPT_METADATA_PATH, "f3_prompt_templates.json")
add_package_file(F3_PROMPT_AUDIT_PATH, "f3_prompt_audit.csv")
add_package_file(F3_RESULT_SCHEMA_PATH, "f3_result_schema.json")

input_files = {
    "inputs/campi_canonical.csv": CAMPI_PATH,
    "inputs/folds.csv": FOLDS_PATH,
    "inputs/nile_subset.lark": GRAMMAR_PATH,
    "inputs/nile_core.py": NILE_CORE_PATH,
    "inputs/nile_metrics.py": NILE_METRICS_PATH,
    "inputs/ir_core.py": IR_CORE_PATH,
    "inputs/ir_schema.json": IR_SCHEMA_PATH,
    "inputs/ir_references.jsonl": IR_REFERENCES_PATH,
    "inputs/rationales_references.jsonl": RATIONALES_PATH,
    "inputs/f0_manifest.json": F0_DIR / "manifest.json",
    "inputs/f1_manifest.json": F1_DIR / "manifest.json",
    "inputs/f2_manifest.json": F2_DIR / "manifest.json",
}

for destination, source in input_files.items():
    add_package_file(source, destination)

for prompt_id in PROMPT_IDS_F3:
    source = F0_DIR / prompt_metadata[prompt_id]["arquivo_relativo"]
    add_package_file(source, f"prompts/{source.name}")


# ----------------------------------------------------------
# 10.3 Manifesto final
# ----------------------------------------------------------

manifest_files = []
for path in sorted(files_to_package):
    manifest_files.append({
        "file": str(path.relative_to(PACOTE_DIR)),
        "size_bytes": path.stat().st_size,
        "sha256": f3_common.sha256_file(path),
    })

F3_MANIFEST = {
    "phase": FASE,
    "description": (
        "Base comum da execução experimental das estratégias F3-A a F3-E."
    ),
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset": DATASET_ID,
    "inputs": {
        "f0_manifest_sha256": HASH_MANIFESTO_F0,
        "f1_manifest_sha256": HASH_MANIFESTO_F1,
        "f1_manifest_sha256_referenced_by_f2": (
            MANIFESTO_F2["inputs"]["f1_manifest_sha256"]
        ),
        "f1_compatibility_mode": pacotes["f1_link_mode"],
        "f1_ir_references_sha256": f3_common.sha256_file(
            F1_DIR / "ir_references.jsonl"
        ),
        "f1_ir_schema_sha256": f3_common.sha256_file(
            F1_DIR / "ir_schema.json"
        ),
        "f2_manifest_sha256": HASH_MANIFESTO_F2,
    },
    "selection": {
        "methods": METODOS_FEWSHOT,
        "k_values": K_FEWSHOT,
        "records_detailed": len(selection_detail_df),
        "records_cumulative": len(selection_jsonl_records),
        "formal_equivalence": FORMAL_EQUIVALENCE_INFO,
        "formal_excluded_train_test_pairs": int(
            FORMAL_EQUIVALENCE_INFO[
                "excluded_train_test_pairs"
            ]
        ),
        "id_leakage_records": int(
            selection_detail_df[
                "vazamento do teste"
            ].sum()
        ),
        "same_nile_records": int(
            selection_detail_df[
                "Nile idêntica à do teste"
            ].sum()
        ),
        "same_ir_records": int(
            selection_detail_df[
                "IR idêntica à do teste"
            ].sum()
        ),
        "formal_equivalence_records": int(
            selection_detail_df[
                "equivalência formal com o teste"
            ].sum()
        ),
        "semantic_model": SEMANTIC_SELECTION_INFO,
    },
    "prompt_audit": PROMPT_AUDIT_INFO,
    "experimental_grid": {
        "configurations": TOTAL_CONFIGURACOES_PRINCIPAIS,
        "main_records": TOTAL_REGISTROS_PRINCIPAIS,
        "final_autocorrection_maximum_eligible_records": (
            TOTAL_REGISTROS_AC_FINAL_MAXIMO
        ),
    },
    "model": MODEL_RUNTIME_INFO,
    "generation": GERACAO,
    "metrics": {
        "names": ["PSR", "EM", "ED", "NED", "SLA-S", "SLA-F"],
        "internal_fields": ["psr", "em", "ed", "ned", "sla_s", "sla_f"],
        "embeddings_metric_used": False,
    },
    "method": {
        "student_model_used_for_experimental_outputs": False,
        "teacher_model_used": False,
        "fine_tuning_used": False,
        "test_references_exposed_to_student": False,
        "formal_equivalent_references_exposed_as_demonstrations": False,
        "selection_uses_references_only_for_deduplication": True,
        "prompt_audit_covers_all_test_examples": True,
        "final_autocorrection_is_separate_strategy": False,
    },
    "environment": {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "packages": package_versions,
    },
    "files": manifest_files,
}

f3_common.write_json(F3_MANIFEST, F3_MANIFEST_PATH)
files_to_package.append(F3_MANIFEST_PATH)


# ----------------------------------------------------------
# 10.4 Criação e verificação do ZIP final
# ----------------------------------------------------------

if F3_ZIP_PATH.exists():
    F3_ZIP_PATH.unlink()

with zipfile.ZipFile(
    F3_ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as package:
    for path in sorted(files_to_package):
        package.write(
            path,
            arcname=str(path.relative_to(PACOTE_DIR)),
        )

with zipfile.ZipFile(F3_ZIP_PATH, "r") as package:
    zip_names = sorted(package.namelist())
    bad_file = package.testzip()

if bad_file is not None:
    raise ValueError(f"Arquivo corrompido dentro do ZIP: {bad_file}")

expected_names = sorted(
    str(path.relative_to(PACOTE_DIR))
    for path in files_to_package
)

if zip_names != expected_names:
    raise ValueError(
        "O conteúdo do ZIP difere da lista esperada."
    )

EXPECTED_PACKAGE_FILES = 30

if len(zip_names) != EXPECTED_PACKAGE_FILES:
    raise ValueError(
        "Quantidade inesperada de arquivos no ZIP: "
        f"{len(zip_names)}. "
        f"Esperado: {EXPECTED_PACKAGE_FILES}."
    )

package_summary_df = pd.DataFrame([
    {
        "grupo": "Módulo",
        "item": "f3_common.py",
        "total": 1,
        "status": "OK",
    },
    {
        "grupo": "Configuração",
        "item": (
            "Configuração, grade, esquema "
            "e metadados dos templates"
        ),
        "total": 4,
        "status": "OK",
    },
    {
        "grupo": "Auditorias",
        "item": (
            "Entradas, equivalência formal "
            "e prompts"
        ),
        "total": 3,
        "status": "OK",
    },
    {
        "grupo": "Seleção",
        "item": "Detalhe e mapa cumulativo",
        "total": 2,
        "status": "OK",
    },
    {
        "grupo": "Entradas",
        "item": (
            "Artefatos congelados de F0, F1 e F2"
        ),
        "total": len(input_files),
        "status": "OK",
    },
    {
        "grupo": "Prompts",
        "item": "Templates da Fase 3",
        "total": len(PROMPT_IDS_F3),
        "status": "OK",
    },
    {
        "grupo": "Pacote",
        "item": "Arquivos no ZIP",
        "total": len(zip_names),
        "status": "OK",
    },
])

exibir_tabela(
    package_summary_df,
    "Conteúdo final da F3-modelo",
    altura_px=360,
)

final_df = pd.DataFrame([{
    "fase": FASE,
    "conjunto": DATASET_ID,
    "exemplos": TOTAL_EXEMPLOS,
    "folds": TOTAL_FOLDS,
    "configurações principais": TOTAL_CONFIGURACOES_PRINCIPAIS,
    "registros principais esperados": TOTAL_REGISTROS_PRINCIPAIS,
    "seleções cumulativas": len(selection_jsonl_records),
    "equivalências formais selecionadas": int(
        selection_detail_df[
            "equivalência formal com o teste"
        ].sum()
    ),
    "prompts auditados": len(prompt_audit_df),
    "vazamentos nos prompts": int(
        prompt_audit_df[
            "vazamento detectado"
        ].sum()
    ),
    "arquivos no pacote": len(zip_names),
    "sha256 do pacote": f3_common.sha256_file(F3_ZIP_PATH),
    "arquivo": str(F3_ZIP_PATH),
    "status": "approved",
}])

exibir_tabela(
    final_df,
    "Resumo final da F3-modelo",
    altura_px=280,
)

print(f"ZIP gerado: {F3_ZIP_PATH}")
print(f"Tamanho: {F3_ZIP_PATH.stat().st_size:,} bytes")
print("F3-modelo concluída com sucesso")
print("Status: OK")

grupo,item,total,status
Módulo,f3_common.py,1,OK
Configuração,"Configuração, grade, esquema e metadados dos templates",4,OK
Auditorias,"Entradas, equivalência formal e prompts",3,OK
Seleção,Detalhe e mapa cumulativo,2,OK
Entradas,"Artefatos congelados de F0, F1 e F2",12,OK
Prompts,Templates da Fase 3,7,OK
Pacote,Arquivos no ZIP,30,OK


fase,conjunto,exemplos,folds,configurações principais,registros principais esperados,seleções cumulativas,equivalências formais selecionadas,prompts auditados,vazamentos nos prompts,arquivos no pacote,sha256 do pacote,arquivo,status
F3-modelo,CAMPI,50,5,74,3700,450,0,4600,0,30,40b260443550d558bf11da520d1424cf9668871592b6c5ea044d5e5251428afb,/kaggle/working/f3_modelo_operacional.zip,approved


ZIP gerado: /kaggle/working/f3_modelo_operacional.zip
Tamanho: 122,579 bytes
F3-modelo concluída com sucesso
Status: OK
